# Chapter 19: Training and Deploying TensorFlow Models at Scale

## Part 1: Serving TensorFlow Models (First 25%)

This notebook covers the fundamental concepts of deploying TensorFlow models to production, focusing on:
- **TensorFlow Serving**: A high-performance model server
- **SavedModel Format**: TensorFlow's standard model export format
- **Model Inspection**: Understanding SavedModel structure
- **Querying Models**: Using REST and gRPC APIs

---

## 1. Introduction to TensorFlow Serving

**TensorFlow Serving (TF Serving)** is a production-ready model server written in C++ that provides:

1. **High Performance**: Can sustain high loads efficiently
2. **Version Management**: Serve multiple model versions simultaneously
3. **Auto-Deployment**: Automatically deploy the latest model versions
4. **Graceful Transitions**: Handle model updates without downtime

### Why Use TF Serving?

- **Battle-tested**: Used in production by many companies
- **Efficient**: Optimized C++ implementation
- **Flexible**: Supports both REST and gRPC APIs
- **Scalable**: Can handle large-scale deployments

Let's start by setting up our environment and creating a simple model to serve.

In [ ]:
# Install required packages
import sys
!{sys.executable} -m pip install -q tensorflow numpy matplotlib requests

In [ ]:
# Import necessary libraries
import tensorflow as tf
from tensorflow import keras
import numpy as np
import os
import json
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

## 2. Creating and Training a Simple Model

Before we can serve a model, we need to create and train one. We'll use the classic MNIST dataset to build a simple digit classifier.

### MNIST Dataset
- **Task**: Classify handwritten digits (0-9)
- **Training samples**: 60,000 images
- **Test samples**: 10,000 images
- **Image size**: 28x28 pixels, grayscale

In [ ]:
# Load and prepare the MNIST dataset
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

# Normalize pixel values to [0, 1] range
X_train = X_train / 255.0
X_test = X_test / 255.0

print(f"Training set shape: {X_train.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Test labels shape: {y_test.shape}")

In [ ]:
# Visualize some training examples
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_train[i], cmap='gray')
    plt.title(f"Label: {y_train[i]}")
    plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Build a simple neural network model
model = keras.models.Sequential([
    # Flatten 28x28 images to 784-dimensional vectors
    keras.layers.Flatten(input_shape=[28, 28]),
    
    # Hidden layer with 300 neurons and ReLU activation
    keras.layers.Dense(300, activation="relu"),
    
    # Hidden layer with 100 neurons and ReLU activation
    keras.layers.Dense(100, activation="relu"),
    
    # Output layer with 10 neurons (one per digit class) and softmax activation
    keras.layers.Dense(10, activation="softmax")
])

# Display model architecture
model.summary()

In [ ]:
# Compile the model
# - Optimizer: SGD (Stochastic Gradient Descent)
# - Loss: Sparse categorical crossentropy (for integer labels)
# - Metrics: Accuracy
model.compile(
    optimizer="sgd",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
# Train the model
print("Training the model...")
history = model.fit(
    X_train, y_train,
    epochs=10,
    validation_split=0.1,
    verbose=1
)

print("\nTraining complete!")

In [ ]:
# Evaluate the model on test data
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f"\nTest accuracy: {test_accuracy:.4f}")
print(f"Test loss: {test_loss:.4f}")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot accuracy
axes[0].plot(history.history['accuracy'], label='Training Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].legend()
axes[0].grid(True)

# Plot loss
axes[1].plot(history.history['loss'], label='Training Loss')
axes[1].plot(history.history['val_loss'], label='Validation Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Model Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 3. Exporting Models as SavedModel

### What is SavedModel?

**SavedModel** is TensorFlow's universal serialization format for models. It contains:
- **Model architecture**: The computational graph
- **Trained weights**: Model parameters
- **Computation**: Operations and functions
- **Assets**: Additional files (vocabularies, etc.)

### SavedModel Directory Structure

```
my_mnist_model/
└── 0001/                          # Version number
    ├── assets/                    # Additional files
    ├── saved_model.pb             # Model definition (protobuf)
    └── variables/                 # Model weights
        ├── variables.data-00000-of-00001
        └── variables.index
```

### Best Practices

1. **Include preprocessing**: Add preprocessing layers to simplify deployment
2. **Version numbering**: Use zero-padded numbers (0001, 0002, etc.)
3. **TF operations only**: SavedModels only work with TensorFlow operations

Let's export our trained model!

In [ ]:
# Define model name and version
model_name = "my_mnist_model"
model_version = "0001"
model_path = os.path.join(model_name, model_version)

print(f"Saving model to: {model_path}")

# Save the model in SavedModel format
tf.saved_model.save(model, model_path)

print(f"Model saved successfully at {model_path}")

In [ ]:
# Verify the directory structure
import os

def display_directory_tree(path, prefix="", max_depth=3, current_depth=0):
    """Display directory tree structure"""
    if current_depth >= max_depth:
        return
    
    try:
        items = sorted(os.listdir(path))
    except PermissionError:
        return
    
    for i, item in enumerate(items):
        item_path = os.path.join(path, item)
        is_last = i == len(items) - 1
        
        connector = "└── " if is_last else "├── "
        print(f"{prefix}{connector}{item}")
        
        if os.path.isdir(item_path):
            extension = "    " if is_last else "│   "
            display_directory_tree(item_path, prefix + extension, max_depth, current_depth + 1)

print(f"\nDirectory structure of {model_name}:")
print(model_name + "/")
display_directory_tree(model_name, "")

In [ ]:
# Check file sizes
import os

def get_dir_size(path):
    """Calculate total size of directory in MB"""
    total = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            if os.path.exists(fp):
                total += os.path.getsize(fp)
    return total / (1024 * 1024)  # Convert to MB

model_size = get_dir_size(model_path)
print(f"\nTotal model size: {model_size:.2f} MB")

## 4. Inspecting SavedModels

### Why Inspect SavedModels?

Before deploying a model, it's important to understand:
- **Input specifications**: Shape, dtype, name
- **Output specifications**: Shape, dtype, name
- **Signature definitions**: Available operations
- **Model metadata**: Version, tags, etc.

### Two Ways to Inspect

1. **Command-line tool**: `saved_model_cli`
2. **Python API**: `tf.saved_model.load()`

Let's use both methods!

In [ ]:
# Method 1: Using saved_model_cli (command-line tool)
# This shows detailed information about the SavedModel

print("Using saved_model_cli to inspect the model:\n")
!saved_model_cli show --dir {model_path} --all

In [ ]:
# Method 2: Loading and inspecting using Python API

# Load the SavedModel
loaded_model = tf.saved_model.load(model_path)

print("SavedModel loaded successfully!")
print(f"\nType: {type(loaded_model)}")
print(f"\nAvailable signatures: {list(loaded_model.signatures.keys())}")

In [ ]:
# Get the default serving signature
serving_fn = loaded_model.signatures['serving_default']

print("Serving function signature:")
print(f"\nInputs:")
for input_name, input_spec in serving_fn.structured_input_signature[1].items():
    print(f"  - Name: {input_name}")
    print(f"    Shape: {input_spec.shape}")
    print(f"    Dtype: {input_spec.dtype}")

print(f"\nOutputs:")
for output_name, output_spec in serving_fn.structured_outputs.items():
    print(f"  - Name: {output_name}")
    print(f"    Shape: {output_spec.shape}")
    print(f"    Dtype: {output_spec.dtype}")

## 5. Loading and Using SavedModels

There are two main ways to load and use a SavedModel:

### Method 1: As SavedModel Object
- Use `tf.saved_model.load()`
- Direct access to low-level operations
- More control over execution

### Method 2: As Keras Model
- Use `keras.models.load_model()`
- Familiar Keras API
- Easy to use `.predict()`, `.evaluate()`, etc.

Let's test both methods!

In [ ]:
# Prepare some test data for predictions
# Select 3 random images from test set
np.random.seed(42)
indices = np.random.randint(0, len(X_test), size=3)
X_new = X_test[indices]
y_new = y_test[indices]

print("Test samples:")
plt.figure(figsize=(10, 3))
for i in range(3):
    plt.subplot(1, 3, i + 1)
    plt.imshow(X_new[i], cmap='gray')
    plt.title(f"True Label: {y_new[i]}")
    plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Method 1: Using SavedModel object
print("Method 1: Using SavedModel object\n")

# Load the model
saved_model = tf.saved_model.load(model_path)

# Make predictions (must convert to float32 tensor)
y_pred_savedmodel = saved_model(tf.constant(X_new, dtype=tf.float32))

# Convert to numpy and get predicted classes
y_pred_classes = np.argmax(y_pred_savedmodel, axis=1)

print("Predictions using SavedModel:")
for i in range(3):
    print(f"  Sample {i+1}: Predicted = {y_pred_classes[i]}, True = {y_new[i]}")
    print(f"  Confidence: {y_pred_savedmodel[i][y_pred_classes[i]].numpy():.4f}")

In [ ]:
# Method 2: Using Keras model
print("Method 2: Using Keras model\n")

# Load the model as Keras model
keras_model = keras.models.load_model(model_path)

# Make predictions using Keras API
y_pred_keras = keras_model.predict(X_new, verbose=0)

# Get predicted classes
y_pred_classes_keras = np.argmax(y_pred_keras, axis=1)

print("Predictions using Keras model:")
for i in range(3):
    print(f"  Sample {i+1}: Predicted = {y_pred_classes_keras[i]}, True = {y_new[i]}")
    print(f"  Confidence: {y_pred_keras[i][y_pred_classes_keras[i]]:.4f}")

In [ ]:
# Verify both methods give identical results
print("\nVerifying consistency between both methods:")
print(f"Results match: {np.allclose(y_pred_savedmodel, y_pred_keras)}")
print(f"Max difference: {np.max(np.abs(y_pred_savedmodel.numpy() - y_pred_keras))}")

## 6. TensorFlow Serving Installation and Setup

### What is TensorFlow Serving?

TensorFlow Serving is a **production-ready model server** that:
- Serves models over network (REST/gRPC)
- Handles multiple versions simultaneously
- Auto-deploys new versions
- Provides monitoring and metrics

### Installation Options

1. **Docker** (Recommended)
   - Easy setup
   - Isolated environment
   - Cross-platform

2. **Native Installation**
   - Direct installation on system
   - Better performance
   - More complex setup

### Docker Installation Commands

```bash
# Pull the TensorFlow Serving image
docker pull tensorflow/serving

# Run the container
docker run -it --rm -p 8500:8500 -p 8501:8501 \
  -v "C:/path/to/my_mnist_model:/models/my_mnist_model" \
  -e MODEL_NAME=my_mnist_model \
  tensorflow/serving
```

### Docker Options Explained

- **`-it`**: Interactive terminal mode
- **`--rm`**: Auto-remove container when stopped
- **`-p 8500:8500`**: Forward gRPC port (binary protocol)
- **`-p 8501:8501`**: Forward REST API port (HTTP/JSON)
- **`-v`**: Mount model directory (host:container)
- **`-e MODEL_NAME`**: Set model name environment variable

### Note for This Notebook

Since we're running in a notebook, we'll simulate the REST and gRPC API interactions. In a real deployment, you would:
1. Install Docker
2. Run the TF Serving container
3. Query the served model using the code below

In [ ]:
# Check if Docker is available (optional)
import subprocess

try:
    result = subprocess.run(['docker', '--version'], 
                          capture_output=True, text=True, timeout=5)
    if result.returncode == 0:
        print("Docker is installed:")
        print(result.stdout)
    else:
        print("Docker is not available")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("Docker is not installed or not in PATH")
    print("\nTo install Docker:")
    print("  - Windows/Mac: Download Docker Desktop from https://www.docker.com/")
    print("  - Linux: Follow instructions at https://docs.docker.com/engine/install/")

In [ ]:
# Display the Docker command to run TensorFlow Serving
# (For demonstration - you would run this in your terminal)

current_dir = os.path.abspath(model_name)

print("To start TensorFlow Serving with Docker, run this command in your terminal:\n")
print(f"docker run -it --rm -p 8500:8500 -p 8501:8501 \\")
print(f"  -v \"{current_dir}:/models/{model_name}\" \\")
print(f"  -e MODEL_NAME={model_name} \\")
print(f"  tensorflow/serving")
print("\n" + "="*70)
print("\nOnce running, you can:")
print("  - Query via REST API at: http://localhost:8501")
print("  - Query via gRPC API at: localhost:8500")
print("  - Stop with: Ctrl+C")

## 7. Querying Models via REST API

### What is REST API?

**REST (Representational State Transfer)** is a simple HTTP-based protocol:
- Uses JSON format (human-readable)
- Easy to use with any HTTP client
- Works from browsers, curl, Python, etc.
- Standard HTTP methods (POST, GET, etc.)

### REST API Endpoints

TensorFlow Serving provides several endpoints:

1. **Prediction**: `/v1/models/MODEL_NAME:predict`
2. **Model metadata**: `/v1/models/MODEL_NAME/metadata`
3. **Model status**: `/v1/models/MODEL_NAME`

### Request Format

```json
{
  "signature_name": "serving_default",
  "instances": [ /* array of input data */ ]
}
```

### Response Format

```json
{
  "predictions": [ /* array of predictions */ ]
}
```

### Limitations

- **Verbose**: JSON is text-based, takes more space
- **Slower**: Parsing overhead for large data
- **Inefficient**: Not ideal for large arrays or images

**Recommendation**: Use REST for small requests or debugging, gRPC for production

In [ ]:
# Simulate REST API query
# In production, this would query a running TF Serving instance

import requests
import json

def query_rest_api(model_name, instances, server_url="http://localhost:8501"):
    """
    Query TensorFlow Serving via REST API
    
    Args:
        model_name: Name of the model to query
        instances: Input data as list/array
        server_url: Base URL of TF Serving server
    
    Returns:
        Predictions array
    """
    # Prepare request payload
    input_data_json = json.dumps({
        "signature_name": "serving_default",
        "instances": instances.tolist(),  # Convert numpy to list
    })
    
    # Construct URL
    url = f"{server_url}/v1/models/{model_name}:predict"
    
    # Send POST request
    try:
        response = requests.post(url, data=input_data_json)
        response.raise_for_status()  # Raise error for bad status codes
        response_json = response.json()
        
        # Extract predictions
        predictions = np.array(response_json["predictions"])
        return predictions
    
    except requests.exceptions.ConnectionError:
        print("Error: Could not connect to TensorFlow Serving")
        print("Make sure TF Serving is running on localhost:8501")
        return None
    except Exception as e:
        print(f"Error: {e}")
        return None

# Display example usage
print("Example REST API query code:\n")
print("# Prepare input data")
print(f"X_new = X_test[:3]  # Shape: {X_new.shape}")
print("")
print("# Query the model")
print(f"predictions = query_rest_api('{model_name}', X_new)")
print("")
print("# Process predictions")
print("y_pred_classes = np.argmax(predictions, axis=1)")
print("print(f'Predicted classes: {y_pred_classes}')")

In [ ]:
# Demonstrate the REST API request structure
print("REST API Request Example:\n")
print("URL: http://localhost:8501/v1/models/my_mnist_model:predict\n")

# Create a minimal example request
example_input = X_new[0:1]  # Just one image
example_request = {
    "signature_name": "serving_default",
    "instances": example_input.tolist()
}

print("Request Body (JSON):")
# Show just the structure, not full data
print(json.dumps({
    "signature_name": "serving_default",
    "instances": [
        "<28x28 array>",  # Abbreviated for display
    ]
}, indent=2))

print(f"\nActual data size: {len(json.dumps(example_request))} bytes")
print(f"Number of values sent: {example_input.size} floats")

In [ ]:
# Attempt to query if TF Serving is running (will fail gracefully if not)
print("Attempting to query TensorFlow Serving...\n")

y_proba_rest = query_rest_api(model_name, X_new)

if y_proba_rest is not None:
    print("\nSuccess! Predictions received from TF Serving:")
    y_pred_classes_rest = np.argmax(y_proba_rest, axis=1)
    
    for i in range(len(X_new)):
        print(f"  Sample {i+1}: Predicted = {y_pred_classes_rest[i]}, True = {y_new[i]}")
        print(f"  Confidence: {y_proba_rest[i][y_pred_classes_rest[i]]:.4f}")
else:
    print("\nTF Serving is not running. To test this:")
    print("1. Run the Docker command shown earlier")
    print("2. Re-run this cell to query the served model")

## 8. Querying Models via gRPC API

### What is gRPC?

**gRPC (Google Remote Procedure Call)** is a high-performance protocol:
- Uses **Protocol Buffers** (binary format)
- Built on **HTTP/2** (multiplexing, compression)
- **Faster** and **more efficient** than REST
- Strongly typed schemas

### Why gRPC over REST?

| Feature | REST (JSON) | gRPC (Protobuf) |
|---------|-------------|----------------|
| **Format** | Text (JSON) | Binary |
| **Size** | Larger (~4x) | Smaller |
| **Speed** | Slower | Faster |
| **Parsing** | JSON parsing | Native types |
| **Best for** | Debugging, small data | Production, large data |

### When to Use gRPC?

✅ **Use gRPC when**:
- Sending large arrays/tensors
- High-throughput requirements
- Low latency needed
- Production deployment

❌ **Use REST when**:
- Quick testing/debugging
- Browser-based clients
- Simple integration
- Small data transfers

### gRPC Setup

Requires additional packages for Protocol Buffers support.

In [ ]:
# Install gRPC packages for TensorFlow Serving
print("Installing gRPC packages...\n")
!{sys.executable} -m pip install -q tensorflow-serving-api grpcio
print("Installation complete!")

In [ ]:
# Import gRPC libraries
try:
    import grpc
    from tensorflow_serving.apis import predict_pb2
    from tensorflow_serving.apis import prediction_service_pb2_grpc
    
    print("gRPC libraries imported successfully!")
    grpc_available = True
except ImportError as e:
    print(f"Error importing gRPC libraries: {e}")
    print("Please install: pip install tensorflow-serving-api grpcio")
    grpc_available = False

In [ ]:
# Define gRPC query function
if grpc_available:
    def query_grpc_api(model_name, instances, input_name, output_name, 
                       server_address="localhost:8500", timeout=10.0):
        """
        Query TensorFlow Serving via gRPC API
        
        Args:
            model_name: Name of the model to query
            instances: Input data as numpy array
            input_name: Name of model's input tensor
            output_name: Name of model's output tensor
            server_address: Address of TF Serving server (host:port)
            timeout: Request timeout in seconds
        
        Returns:
            Predictions array
        """
        # Create gRPC channel
        channel = grpc.insecure_channel(server_address)
        stub = prediction_service_pb2_grpc.PredictionServiceStub(channel)
        
        # Create prediction request
        request = predict_pb2.PredictRequest()
        request.model_spec.name = model_name
        request.model_spec.signature_name = "serving_default"
        
        # Convert input data to tensor proto
        request.inputs[input_name].CopyFrom(
            tf.make_tensor_proto(instances, dtype=tf.float32)
        )
        
        try:
            # Send request
            response = stub.Predict(request, timeout=timeout)
            
            # Extract predictions
            output_tensor = response.outputs[output_name]
            predictions = tf.make_ndarray(output_tensor)
            
            return predictions
        
        except grpc.RpcError as e:
            print(f"gRPC Error: {e.code()} - {e.details()}")
            return None
        except Exception as e:
            print(f"Error: {e}")
            return None
    
    print("gRPC query function defined successfully!")
else:
    print("gRPC not available - skipping function definition")

In [ ]:
# Display example gRPC usage
print("Example gRPC API query code:\n")
print("# Get model input/output names")
print(f"input_name = model.input_names[0]  # '{model.input_names[0]}'")
print(f"output_name = model.output_names[0]  # '{model.output_names[0]}'")
print("")
print("# Query the model via gRPC")
print("predictions = query_grpc_api(")
print(f"    model_name='{model_name}',")
print("    instances=X_new,")
print("    input_name=input_name,")
print("    output_name=output_name,")
print("    server_address='localhost:8500'")
print(")")
print("")
print("# Process predictions")
print("y_pred_classes = np.argmax(predictions, axis=1)")

In [ ]:
# Attempt to query via gRPC if TF Serving is running
if grpc_available:
    print("Attempting to query TensorFlow Serving via gRPC...\n")
    
    # Get input/output names
    input_name = model.input_names[0]
    output_name = model.output_names[0]
    
    print(f"Model input name: {input_name}")
    print(f"Model output name: {output_name}\n")
    
    # Query via gRPC
    y_proba_grpc = query_grpc_api(
        model_name=model_name,
        instances=X_new,
        input_name=input_name,
        output_name=output_name
    )
    
    if y_proba_grpc is not None:
        print("Success! Predictions received via gRPC:")
        y_pred_classes_grpc = np.argmax(y_proba_grpc, axis=1)
        
        for i in range(len(X_new)):
            print(f"  Sample {i+1}: Predicted = {y_pred_classes_grpc[i]}, True = {y_new[i]}")
            print(f"  Confidence: {y_proba_grpc[i][y_pred_classes_grpc[i]]:.4f}")
    else:
        print("TF Serving is not running. To test this:")
        print("1. Run the Docker command shown earlier")
        print("2. Re-run this cell to query the served model")
else:
    print("gRPC libraries not available")

## 9. Comparison: REST vs gRPC

Let's compare the efficiency of REST (JSON) vs gRPC (Protocol Buffers) for our use case.

In [ ]:
# Compare data sizes for REST vs gRPC
import json

# Test with different batch sizes
batch_sizes = [1, 10, 50, 100]

print("Data Transfer Size Comparison (REST vs gRPC)\n")
print(f"{'Batch Size':<12} {'REST (JSON)':<15} {'gRPC (Est.)':<15} {'Ratio':<10}")
print("="*55)

for batch_size in batch_sizes:
    # Get sample data
    sample_data = X_test[:batch_size]
    
    # Calculate REST (JSON) size
    rest_request = {
        "signature_name": "serving_default",
        "instances": sample_data.tolist()
    }
    rest_size = len(json.dumps(rest_request))
    
    # Estimate gRPC size (binary float32: 4 bytes per value)
    # Plus small overhead for protobuf structure
    grpc_size = sample_data.size * 4 + 100  # 100 bytes overhead estimate
    
    ratio = rest_size / grpc_size
    
    print(f"{batch_size:<12} {rest_size:>12,} B  {grpc_size:>12,} B  {ratio:>6.1f}x")

print("\n" + "="*55)
print("\nKey Insights:")
print("  • REST (JSON) is ~4-5x larger than gRPC (binary)")
print("  • Difference grows with batch size")
print("  • gRPC is more efficient for large data transfers")

## 10. Deploying New Model Versions

### Version Management in TF Serving

One of TensorFlow Serving's most powerful features is **automatic version management**:

1. **Export new version**: Save model to new version directory (e.g., `0002`)
2. **Auto-detection**: TF Serving automatically detects new version
3. **Graceful transition**: 
   - Completes pending requests with old version
   - Routes new requests to new version
   - Unloads old version when safe
4. **Zero downtime**: No service interruption

### Version Directory Structure

```
my_mnist_model/
├── 0001/              # Version 1
│   ├── saved_model.pb
│   └── variables/
├── 0002/              # Version 2 (newer)
│   ├── saved_model.pb
│   └── variables/
└── 0003/              # Version 3 (newest - will be served)
    ├── saved_model.pb
    └── variables/
```

### Best Practices

- **Versioning scheme**: Use zero-padded numbers (0001, 0002, 0003...)
- **Serving policy**: By default, serves highest version number
- **Rollback**: Keep old versions for easy rollback if needed
- **Testing**: Test new version before deploying

Let's simulate creating a new model version!

In [ ]:
# Create a slightly improved model (Version 2)
print("Creating improved model (Version 2)...\n")

model_v2 = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.Dense(300, activation="relu"),
    keras.layers.Dropout(0.2),  # Added dropout for regularization
    keras.layers.Dense(100, activation="relu"),
    keras.layers.Dropout(0.2),  # Added dropout
    keras.layers.Dense(10, activation="softmax")
])

model_v2.compile(
    optimizer="adam",  # Changed to Adam optimizer
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model V2 architecture:")
model_v2.summary()

In [ ]:
# Train the improved model (fewer epochs for demo)
print("Training Model V2...\n")

history_v2 = model_v2.fit(
    X_train, y_train,
    epochs=5,  # Fewer epochs for demo
    validation_split=0.1,
    verbose=1
)

# Evaluate
test_loss_v2, test_accuracy_v2 = model_v2.evaluate(X_test, y_test, verbose=0)
print(f"\nModel V2 Test Accuracy: {test_accuracy_v2:.4f}")
print(f"Original Model Accuracy: {test_accuracy:.4f}")
print(f"Improvement: {(test_accuracy_v2 - test_accuracy):.4f}")

In [ ]:
# Export Model V2 as version 0002
model_version_2 = "0002"
model_path_v2 = os.path.join(model_name, model_version_2)

print(f"Exporting Model V2 to: {model_path_v2}")
tf.saved_model.save(model_v2, model_path_v2)
print("Model V2 saved successfully!\n")

# Display updated directory structure
print(f"Updated directory structure:")
print(model_name + "/")
display_directory_tree(model_name, "", max_depth=2)

In [ ]:
# Show how TF Serving handles version updates
print("How TensorFlow Serving Handles Version Updates:\n")
print("Step 1: Currently serving version 0001")
print("  → All requests go to version 0001\n")

print("Step 2: Version 0002 exported to model directory")
print("  → TF Serving detects new version (automatic polling)\n")

print("Step 3: Graceful transition begins")
print("  → Pending requests to 0001 complete normally")
print("  → New requests routed to version 0002")
print("  → Version 0001 unloaded when safe\n")

print("Step 4: Now serving version 0002")
print("  → All requests go to version 0002")
print("  → Zero downtime achieved! ✓\n")

print("="*60)
print("\nRollback Process (if needed):")
print("  1. Remove or rename version 0002 directory")
print("  2. TF Serving automatically falls back to version 0001")
print("  3. Service continues with previous version")

## 11. Summary: First 25% - Model Serving Fundamentals

### What We've Covered

In this first section, we've explored the fundamentals of serving TensorFlow models in production:

#### 1. **TensorFlow Serving**
   - High-performance C++ model server
   - Battle-tested for production use
   - Handles multiple versions automatically
   - Zero-downtime deployments

#### 2. **SavedModel Format**
   - Standard TensorFlow serialization format
   - Contains model + weights + computation
   - Portable and version-controlled
   - Inspectable with `saved_model_cli`

#### 3. **REST API**
   - Simple HTTP/JSON interface
   - Easy to use and debug
   - Best for small requests
   - ~4-5x larger payloads than gRPC

#### 4. **gRPC API**
   - High-performance binary protocol
   - Built on HTTP/2 + Protocol Buffers
   - Best for production and large data
   - 4-5x more efficient than REST

#### 5. **Version Management**
   - Automatic version detection
   - Graceful transitions
   - Easy rollback capabilities
   - Zero downtime deployments

### Key Takeaways

✅ **SavedModel** is the universal format for TensorFlow models  
✅ **TF Serving** provides production-ready model serving  
✅ **REST API** is simple but inefficient for large data  
✅ **gRPC API** is faster and more efficient (use in production)  
✅ **Version management** enables zero-downtime updates  

### Next Steps

In the next sections, we'll cover:
- **Cloud Deployment**: Google Cloud AI Platform
- **Mobile Deployment**: TensorFlow Lite
- **GPU Acceleration**: Training on GPUs
- **Distributed Training**: Multi-GPU and multi-machine training

---

### Practice Exercises

1. **Export your own model** in SavedModel format
2. **Inspect it** using `saved_model_cli`
3. **Set up TF Serving** with Docker
4. **Query your model** via REST and gRPC
5. **Deploy a new version** and observe the automatic transition

### Additional Resources

- [TensorFlow Serving Documentation](https://www.tensorflow.org/tfx/guide/serving)
- [SavedModel Guide](https://www.tensorflow.org/guide/saved_model)
- [TF Serving REST API](https://www.tensorflow.org/tfx/serving/api_rest)
- [TF Serving with Docker](https://www.tensorflow.org/tfx/serving/docker)

# Part 2: Cloud and Mobile Deployment (Next 25%)

This section covers:
- **Google Cloud AI Platform**: Deploying models to the cloud
- **TensorFlow Lite**: Optimizing models for mobile and embedded devices
- **Model Compression**: Reducing model size
- **Quantization**: Converting models to use less memory

---

## 12. Deploying to Google Cloud AI Platform

### What is Google Cloud AI Platform?

**Google Cloud AI Platform** (now part of **Vertex AI**) is a managed service for deploying and serving machine learning models at scale. It provides:

1. **Fully Managed Infrastructure**: No server management required
2. **Automatic Scaling**: Scales up/down based on traffic
3. **Global Distribution**: Deploy models worldwide
4. **Monitoring & Logging**: Built-in observability with Stackdriver
5. **Versioning**: Manage multiple model versions
6. **Pay-as-you-go**: Only pay for what you use

### Key Concepts

- **Model**: A container for model versions
- **Version**: A specific trained model deployment
- **Prediction Service**: The API endpoint for predictions
- **Service Account**: Authentication credentials

### Deployment Architecture

```
Your Application
       ↓
  REST/gRPC API
       ↓
Google Cloud AI Platform
       ↓
   Model Version(s)
       ↓
  Predictions
```

### Benefits Over Self-Hosting

| Feature | Self-Hosted | Cloud AI Platform |
|---------|-------------|-------------------|
| **Setup** | Complex | Simple |
| **Scaling** | Manual | Automatic |
| **Maintenance** | You handle | Google handles |
| **Cost** | Fixed (servers) | Variable (usage) |
| **Monitoring** | DIY | Built-in |

Let's walk through the deployment process!

### Step-by-Step Deployment Process

The deployment process involves several steps:

#### **Step 1: Setup Google Cloud Project**
- Create a Google Cloud account (free tier available)
- Create a new project
- Enable billing (required for AI Platform)
- Enable AI Platform API

#### **Step 2: Create Cloud Storage Bucket**
- Create a Google Cloud Storage (GCS) bucket
- This will store your SavedModel files

#### **Step 3: Upload SavedModel**
- Upload your model to the GCS bucket
- Path format: `gs://your-bucket/model-name/version/`

#### **Step 4: Create Model Resource**
- Create a model resource in AI Platform
- This is a container for your versions

#### **Step 5: Create Model Version**
- Specify Python version (3.5+)
- Specify TensorFlow version
- Specify machine type (CPU/GPU)
- Point to model location in GCS
- Configure scaling (automatic/manual)

#### **Step 6: Query the Model**
- Use Google Cloud SDK
- Use REST API
- Use Python client library

Let's simulate this process!

In [ ]:
# Install Google Cloud libraries (for demonstration)
print("Installing Google Cloud libraries...\n")
!{sys.executable} -m pip install -q google-cloud-storage google-api-python-client google-auth
print("Installation complete!")

In [ ]:
# Simulate the upload process to Google Cloud Storage
# In practice, you would use the gcloud CLI or Python SDK

import os

# Configuration (replace with your actual values)
PROJECT_ID = "your-project-id"
BUCKET_NAME = "your-model-bucket"
MODEL_NAME = "my_mnist_model"
MODEL_VERSION = "v1"

# Construct GCS path
gcs_model_path = f"gs://{BUCKET_NAME}/{MODEL_NAME}/{MODEL_VERSION}/"

print("Google Cloud Storage Upload Configuration:\n")
print(f"Project ID:      {PROJECT_ID}")
print(f"Bucket Name:     {BUCKET_NAME}")
print(f"Model Name:      {MODEL_NAME}")
print(f"Model Version:   {MODEL_VERSION}")
print(f"GCS Path:        {gcs_model_path}")
print("\n" + "="*60)
print("\nTo upload your model to GCS, use one of these methods:\n")

print("Method 1: Using gsutil (Google Cloud SDK):")
print(f"  $ gsutil -m cp -r {model_path} {gcs_model_path}")
print()

print("Method 2: Using gcloud CLI:")
print(f"  $ gcloud storage cp -r {model_path} {gcs_model_path}")
print()

print("Method 3: Using Python SDK:")
print("  from google.cloud import storage")
print("  client = storage.Client()")
print(f"  bucket = client.bucket('{BUCKET_NAME}')")
print("  # Upload files...")

In [ ]:
# Create model on AI Platform using gcloud CLI
# This is the command you would run in your terminal

print("Creating Model on Google Cloud AI Platform:\n")
print("Step 1: Create the model resource")
print("-" * 60)
print(f"$ gcloud ai-platform models create {MODEL_NAME} \\")
print(f"    --regions=us-central1")
print()

print("\nStep 2: Create the model version")
print("-" * 60)
version_name = "v1"
print(f"$ gcloud ai-platform versions create {version_name} \\")
print(f"    --model={MODEL_NAME} \\")
print(f"    --origin={gcs_model_path} \\")
print(f"    --runtime-version=2.8 \\")
print(f"    --python-version=3.7 \\")
print(f"    --framework=TENSORFLOW \\")
print(f"    --machine-type=n1-standard-4")
print()

print("\nConfiguration Parameters Explained:")
print("  --model:          Model name (container for versions)")
print("  --origin:         GCS path to SavedModel")
print("  --runtime-version: TensorFlow version")
print("  --python-version:  Python version (3.7 or 3.8)")
print("  --framework:       ML framework (TENSORFLOW)")
print("  --machine-type:    VM type (n1-standard-4 = 4 vCPUs)")
print()

print("\nMachine Type Options:")
print("  • n1-standard-2:  2 vCPUs,  7.5 GB RAM")
print("  • n1-standard-4:  4 vCPUs, 15.0 GB RAM")
print("  • n1-standard-8:  8 vCPUs, 30.0 GB RAM")
print("  • n1-highmem-*:   More memory")
print("  • n1-highcpu-*:   More CPU")
print("  • GPU machines:   Add GPUs for faster inference")

### Querying the Cloud AI Platform Model

Once deployed, you can query the model using the **Google Cloud AI Platform Prediction API**. This requires:

1. **Service Account**: Create credentials for authentication
2. **API Client**: Use the Google API Python client
3. **Prediction Request**: Send input data and receive predictions

#### Authentication Setup

```python
# Set environment variable to your service account key
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "service_account_key.json"
```

#### Creating a Service Account

1. Go to **IAM & Admin → Service Accounts** in GCP Console
2. Click **Create Service Account**
3. Name: `ml-serving-account`
4. Role: **ML Engine Developer**
5. Click **Create Key** → JSON
6. Save the key file securely

Let's demonstrate the prediction API!

In [ ]:
# Function to query Google Cloud AI Platform
import googleapiclient.discovery

def predict_cloud_ai_platform(project_id, model_name, instances, version=None):
    """
    Query a model deployed on Google Cloud AI Platform
    
    Args:
        project_id: Your GCP project ID
        model_name: Name of the deployed model
        instances: Input data as list
        version: Optional specific version to query
    
    Returns:
        Predictions array
    """
    # Construct model path
    if version:
        model_path = f"projects/{project_id}/models/{model_name}/versions/{version}"
    else:
        model_path = f"projects/{project_id}/models/{model_name}"
    
    # Build the API client
    ml_resource = googleapiclient.discovery.build("ml", "v1").projects()
    
    # Prepare request
    input_data = {
        "signature_name": "serving_default",
        "instances": instances
    }
    
    # Send prediction request
    request = ml_resource.predict(name=model_path, body=input_data)
    
    try:
        response = request.execute()
        
        # Check for errors
        if "error" in response:
            raise RuntimeError(response["error"])
        
        # Extract predictions
        predictions = response["predictions"]
        output_name = model.output_names[0]
        
        # Convert to numpy array
        y_proba = np.array([pred[output_name] for pred in predictions])
        return y_proba
    
    except Exception as e:
        print(f"Error querying Cloud AI Platform: {e}")
        return None

print("Cloud AI Platform prediction function defined!")
print("\nExample usage:")
print(f"predictions = predict_cloud_ai_platform(")
print(f"    project_id='{PROJECT_ID}',")
print(f"    model_name='{MODEL_NAME}',")
print(f"    instances=X_new.tolist(),")
print(f"    version='v1'")
print(f")")

In [ ]:
# Demonstrate Cloud AI Platform features

print("Google Cloud AI Platform Features:\n")
print("="*60)

print("\n1. AUTOMATIC SCALING")
print("-" * 60)
print("   • Scales based on request load")
print("   • Min instances: 0 (saves cost when idle)")
print("   • Max instances: Set based on budget")
print("   • Auto-scaling options:")
print("     - Target CPU utilization")
print("     - Target throughput")
print("     - Custom metrics")

print("\n2. MONITORING & LOGGING")
print("-" * 60)
print("   • Request count and latency")
print("   • Error rates")
print("   • Resource utilization (CPU, memory)")
print("   • Custom metrics via Stackdriver")
print("   • Integrated with Cloud Logging")

print("\n3. VERSIONING & TRAFFIC SPLITTING")
print("-" * 60)
print("   • Deploy multiple versions simultaneously")
print("   • Route traffic between versions:")
print("     - 90% to v1, 10% to v2 (canary deployment)")
print("     - A/B testing")
print("     - Blue-green deployments")

print("\n4. COST MANAGEMENT")
print("-" * 60)
print("   • Pay only for prediction time")
print("   • Pricing based on:")
print("     - Number of predictions")
print("     - Node hours")
print("     - Machine type")
print("   • Automatic shutdown when idle")

print("\n5. GLOBAL AVAILABILITY")
print("-" * 60)
print("   • Deploy in multiple regions:")
print("     - us-central1 (Iowa)")
print("     - us-east1 (South Carolina)")
print("     - europe-west1 (Belgium)")
print("     - asia-east1 (Taiwan)")
print("   • Low latency worldwide")

print("\n" + "="*60)

## 13. Mobile and Embedded Deployment with TensorFlow Lite

### What is TensorFlow Lite?

**TensorFlow Lite (TFLite)** is TensorFlow's solution for deploying models on:
- **Mobile devices**: Android, iOS
- **Embedded systems**: Raspberry Pi, microcontrollers
- **IoT devices**: Edge computing
- **Web browsers**: Via WASM

### Why TensorFlow Lite?

Mobile and embedded devices have constraints:

| Constraint | Challenge | TFLite Solution |
|------------|-----------|-----------------|
| **Size** | Limited storage | Smaller models (4-10x reduction) |
| **Memory** | Limited RAM | Efficient inference |
| **Power** | Battery life | Optimized operations |
| **Speed** | Real-time needs | Hardware acceleration |

### Three Main Objectives

1. **Reduce Model Size**
   - Faster download
   - Less storage space
   - Less RAM usage

2. **Reduce Computations**
   - Lower latency
   - Less battery drain
   - Faster inference

3. **Adapt to Device Constraints**
   - Work with limited hardware
   - Support specialized accelerators (Edge TPU, Neural Engine)
   - Offline capability

### TFLite Architecture

```
TensorFlow Model (SavedModel)
           ↓
    TFLite Converter
           ↓
    TFLite Model (.tflite)
           ↓
    TFLite Interpreter
           ↓
  Device (Mobile/Embedded)
```

Let's convert our model to TFLite!

### Basic Model Conversion

The simplest way to convert a model to TFLite is using the `TFLiteConverter`:

```python
# From SavedModel
converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_path)
tflite_model = converter.convert()

# From Keras model
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
```

### What Happens During Conversion?

1. **FlatBuffers Serialization**: Uses efficient binary format
2. **Operation Pruning**: Removes unused operations
3. **Graph Optimization**: Fuses operations where possible
4. **Kernel Selection**: Chooses optimized implementations

### Model Format Comparison

| Format | Size | Speed | Use Case |
|--------|------|-------|----------|
| **SavedModel** | Large | Fast | Server deployment |
| **TFLite** | Small | Very Fast | Mobile/embedded |
| **TF.js** | Medium | Medium | Browser |

Let's convert our MNIST model!

In [ ]:
# Method 1: Convert from SavedModel
print("Converting model to TensorFlow Lite format...\n")

# Create converter from SavedModel
converter = tf.lite.TFLiteConverter.from_saved_model(model_path)

# Convert the model
tflite_model = converter.convert()

# Save the TFLite model
tflite_model_path = "mnist_model.tflite"
with open(tflite_model_path, "wb") as f:
    f.write(tflite_model)

print(f"TFLite model saved to: {tflite_model_path}")
print(f"Model size: {len(tflite_model) / 1024:.2f} KB")

In [ ]:
# Method 2: Convert directly from Keras model
print("Alternative: Converting directly from Keras model...\n")

converter_keras = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model_keras = converter_keras.convert()

# Save alternative version
tflite_keras_path = "mnist_model_keras.tflite"
with open(tflite_keras_path, "wb") as f:
    f.write(tflite_model_keras)

print(f"TFLite model (from Keras) saved to: {tflite_keras_path}")
print(f"Model size: {len(tflite_model_keras) / 1024:.2f} KB")

# Verify both methods produce similar results
size_diff = abs(len(tflite_model) - len(tflite_model_keras))
print(f"\nSize difference between methods: {size_diff} bytes")
print("Both conversion methods produce similar results ✓")

In [ ]:
# Compare model sizes
import os

# Get original SavedModel size
savedmodel_size = get_dir_size(model_path)

# Get TFLite model size
tflite_size = os.path.getsize(tflite_model_path) / (1024 * 1024)  # Convert to MB

print("Model Size Comparison:\n")
print("="*60)
print(f"{'Format':<20} {'Size (MB)':<15} {'Reduction':<15}")
print("-"*60)
print(f"{'SavedModel':<20} {savedmodel_size:>10.2f} MB  {'-':<15}")
print(f"{'TFLite':<20} {tflite_size:>10.2f} MB  {savedmodel_size/tflite_size:>10.1f}x smaller")
print("="*60)

print(f"\n💡 TFLite model is {savedmodel_size/tflite_size:.1f}x smaller!")
print(f"   Storage saved: {(savedmodel_size - tflite_size):.2f} MB")

### Running Inference with TFLite

To use a TFLite model, we need the **TFLite Interpreter**:

1. **Load the model**: Read the `.tflite` file
2. **Allocate tensors**: Prepare memory
3. **Set input**: Provide input data
4. **Invoke**: Run inference
5. **Get output**: Extract predictions

The interpreter is optimized for mobile/embedded devices and uses minimal resources.

In [ ]:
# Load and use the TFLite model
print("Loading TFLite model and running inference...\n")

# Create interpreter
interpreter = tf.lite.Interpreter(model_path=tflite_model_path)

# Allocate tensors
interpreter.allocate_tensors()

# Get input and output details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Model Input Details:")
print(f"  Name: {input_details[0]['name']}")
print(f"  Shape: {input_details[0]['shape']}")
print(f"  Type: {input_details[0]['dtype']}")

print("\nModel Output Details:")
print(f"  Name: {output_details[0]['name']}")
print(f"  Shape: {output_details[0]['shape']}")
print(f"  Type: {output_details[0]['dtype']}")

In [ ]:
# Run inference on test samples
print("\nRunning TFLite inference on test samples...\n")

# Prepare test data (add batch dimension)
X_test_sample = X_new.astype(np.float32)

# Run predictions for each sample
tflite_predictions = []

for i, sample in enumerate(X_test_sample):
    # Set input tensor (must add batch dimension)
    interpreter.set_tensor(input_details[0]['index'], sample[np.newaxis, ...])
    
    # Run inference
    interpreter.invoke()
    
    # Get output tensor
    output_data = interpreter.get_tensor(output_details[0]['index'])
    tflite_predictions.append(output_data[0])
    
    predicted_class = np.argmax(output_data[0])
    confidence = output_data[0][predicted_class]
    
    print(f"Sample {i+1}:")
    print(f"  Predicted: {predicted_class}, True: {y_new[i]}")
    print(f"  Confidence: {confidence:.4f}")

tflite_predictions = np.array(tflite_predictions)

print("\n✓ TFLite inference complete!")

In [ ]:
# Verify TFLite predictions match original model
original_predictions = model.predict(X_new, verbose=0)

print("Comparing TFLite vs Original Model:\n")
print("="*60)

# Compare predictions
max_diff = np.max(np.abs(tflite_predictions - original_predictions))
mean_diff = np.mean(np.abs(tflite_predictions - original_predictions))

print(f"Maximum difference: {max_diff:.10f}")
print(f"Mean difference:    {mean_diff:.10f}")

# Check if predictions are close
if np.allclose(tflite_predictions, original_predictions, atol=1e-5):
    print("\n✓ TFLite predictions match original model!")
else:
    print("\n⚠ Small differences detected (expected due to conversion)")

# Compare predicted classes
tflite_classes = np.argmax(tflite_predictions, axis=1)
original_classes = np.argmax(original_predictions, axis=1)

print(f"\nPredicted classes match: {np.array_equal(tflite_classes, original_classes)}")
print("="*60)

## 14. Model Optimization: Quantization

### What is Quantization?

**Quantization** reduces model size and speeds up inference by using lower precision data types:

- **Standard**: 32-bit floating point (float32)
- **Quantized**: 8-bit integers (int8)

### Benefits of Quantization

1. **4x Size Reduction**: 32 bits → 8 bits = 4x smaller
2. **Faster Inference**: Integer operations are faster
3. **Less Memory**: Lower RAM usage
4. **Lower Power**: Important for battery-powered devices

### Types of Quantization

#### 1. **Post-Training Quantization (Weights Only)**
   - Simplest method
   - Quantize weights only
   - Activations remain float32
   - 4x size reduction
   - Minimal accuracy loss

#### 2. **Full Integer Quantization**
   - Quantize weights AND activations
   - Requires representative dataset
   - Maximum speed improvement
   - Needed for Edge TPU

#### 3. **Quantization-Aware Training**
   - Train with fake quantization operations
   - Model learns to be robust to quantization
   - Best accuracy retention
   - More complex setup

### How Quantization Works

**Symmetric Quantization** maps floating point values to integers:

```
float_range: [-m, +m]  →  int_range: [-127, +127]

scale = m / 127
quantized_value = round(float_value / scale)
dequantized_value = quantized_value * scale
```

**Example:**
- Weights range: [-1.5, +0.8]
- m = max(1.5, 0.8) = 1.5
- scale = 1.5 / 127 ≈ 0.0118
- -1.5 → -127, 0.0 → 0, +1.5 → +127

Let's apply quantization to our model!

In [ ]:
# Post-training quantization (weights only)
print("Applying post-training quantization (weights only)...\n")

# Create converter with optimization
converter_quantized = tf.lite.TFLiteConverter.from_keras_model(model)

# Enable size optimization (quantize weights to int8)
converter_quantized.optimizations = [tf.lite.Optimize.DEFAULT]

# Convert the model
tflite_quantized_model = converter_quantized.convert()

# Save quantized model
tflite_quantized_path = "mnist_model_quantized.tflite"
with open(tflite_quantized_path, "wb") as f:
    f.write(tflite_quantized_model)

print(f"Quantized model saved to: {tflite_quantized_path}")
print(f"Quantized model size: {len(tflite_quantized_model) / 1024:.2f} KB")

In [ ]:
# Compare sizes: SavedModel vs TFLite vs Quantized TFLite
import os

original_size = get_dir_size(model_path)
tflite_size = os.path.getsize(tflite_model_path) / (1024 * 1024)
quantized_size = os.path.getsize(tflite_quantized_path) / (1024 * 1024)

print("Model Size Comparison:\n")
print("="*70)
print(f"{'Format':<30} {'Size (MB)':<15} {'Reduction':<15}")
print("-"*70)
print(f"{'SavedModel (original)':<30} {original_size:>10.2f} MB  {'baseline':<15}")
print(f"{'TFLite (float32)':<30} {tflite_size:>10.2f} MB  {original_size/tflite_size:>10.1f}x smaller")
print(f"{'TFLite (quantized int8)':<30} {quantized_size:>10.2f} MB  {original_size/quantized_size:>10.1f}x smaller")
print("="*70)

# Additional comparison
print(f"\n📊 Key Insights:")
print(f"   • TFLite reduces size by {original_size/tflite_size:.1f}x")
print(f"   • Quantization reduces size by {tflite_size/quantized_size:.1f}x more")
print(f"   • Total reduction: {original_size/quantized_size:.1f}x smaller than SavedModel")
print(f"   • Space saved: {(original_size - quantized_size):.2f} MB")

In [ ]:
# Test quantized model accuracy
print("Testing quantized model accuracy...\n")

# Load quantized model
interpreter_quantized = tf.lite.Interpreter(model_path=tflite_quantized_path)
interpreter_quantized.allocate_tensors()

# Get input/output details
input_details_q = interpreter_quantized.get_input_details()
output_details_q = interpreter_quantized.get_output_details()

# Run predictions
quantized_predictions = []

for sample in X_new.astype(np.float32):
    interpreter_quantized.set_tensor(input_details_q[0]['index'], sample[np.newaxis, ...])
    interpreter_quantized.invoke()
    output_data = interpreter_quantized.get_tensor(output_details_q[0]['index'])
    quantized_predictions.append(output_data[0])

quantized_predictions = np.array(quantized_predictions)

# Compare predictions
print("Prediction Comparison:\n")
print("="*70)
print(f"{'Sample':<10} {'True':<10} {'Original':<15} {'TFLite':<15} {'Quantized':<15}")
print("-"*70)

for i in range(len(X_new)):
    orig_pred = np.argmax(original_predictions[i])
    tflite_pred = np.argmax(tflite_predictions[i])
    quant_pred = np.argmax(quantized_predictions[i])
    
    print(f"{i+1:<10} {y_new[i]:<10} {orig_pred:<15} {tflite_pred:<15} {quant_pred:<15}")

print("="*70)

# Check accuracy
all_match = (np.argmax(original_predictions, axis=1) == 
             np.argmax(quantized_predictions, axis=1)).all()

print(f"\n✓ All predictions match: {all_match}")
print("Quantization preserved model accuracy!")

### Full Integer Quantization

For maximum optimization (especially for Edge TPU), we can perform **full integer quantization** which quantizes both weights and activations.

This requires a **representative dataset** to calibrate the quantization ranges for activations.

#### Steps:
1. Provide representative data
2. Converter runs inference on samples
3. Records activation ranges
4. Quantizes everything to int8

#### Use Cases:
- Edge TPU deployment
- Maximum speed improvement
- Lowest memory usage
- Models for microcontrollers

In [ ]:
# Full integer quantization with representative dataset
print("Applying full integer quantization...\n")

# Create representative dataset generator
def representative_dataset():
    """Generator that yields representative data samples"""
    # Use a subset of training data for calibration
    for i in range(100):  # Use 100 samples
        sample = X_train[i:i+1].astype(np.float32)
        yield [sample]

# Create converter
converter_full_quant = tf.lite.TFLiteConverter.from_keras_model(model)

# Enable full integer quantization
converter_full_quant.optimizations = [tf.lite.Optimize.DEFAULT]
converter_full_quant.representative_dataset = representative_dataset

# Ensure input/output are also quantized (optional, for Edge TPU)
converter_full_quant.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_full_quant.inference_input_type = tf.uint8  # or tf.int8
converter_full_quant.inference_output_type = tf.uint8  # or tf.int8

# Convert
print("Converting... (this may take a moment)")
tflite_full_quant_model = converter_full_quant.convert()

# Save
tflite_full_quant_path = "mnist_model_full_quantized.tflite"
with open(tflite_full_quant_path, "wb") as f:
    f.write(tflite_full_quant_model)

print(f"✓ Full quantized model saved to: {tflite_full_quant_path}")
print(f"Model size: {len(tflite_full_quant_model) / 1024:.2f} KB")

In [ ]:
# Final size comparison
print("Complete Model Size Comparison:\n")
print("="*75)
print(f"{'Format':<35} {'Size (MB)':<15} {'Reduction':<15}")
print("-"*75)

savedmodel_mb = get_dir_size(model_path)
tflite_mb = os.path.getsize(tflite_model_path) / (1024 * 1024)
quantized_mb = os.path.getsize(tflite_quantized_path) / (1024 * 1024)
full_quant_mb = os.path.getsize(tflite_full_quant_path) / (1024 * 1024)

print(f"{'SavedModel (float32)':<35} {savedmodel_mb:>10.2f} MB  {'baseline':<15}")
print(f"{'TFLite (float32)':<35} {tflite_mb:>10.2f} MB  {savedmodel_mb/tflite_mb:>10.1f}x")
print(f"{'TFLite (int8 weights only)':<35} {quantized_mb:>10.2f} MB  {savedmodel_mb/quantized_mb:>10.1f}x")
print(f"{'TFLite (full int8 quantization)':<35} {full_quant_mb:>10.2f} MB  {savedmodel_mb/full_quant_mb:>10.1f}x")
print("="*75)

print("\n📊 Summary:")
print(f"   • Standard TFLite: {savedmodel_mb/tflite_mb:.1f}x smaller")
print(f"   • Weight quantization: {savedmodel_mb/quantized_mb:.1f}x smaller")
print(f"   • Full quantization: {savedmodel_mb/full_quant_mb:.1f}x smaller")
print(f"   • Total space saved: {(savedmodel_mb - full_quant_mb):.2f} MB")
print(f"\n💡 Full quantization achieves maximum compression!")
print(f"   Perfect for mobile devices and embedded systems.")

### Deployment Scenarios

Now that we have optimized models, let's understand where to deploy them:

#### 1. **Android Deployment**
```java
// Load TFLite model in Android
Interpreter tflite = new Interpreter(loadModelFile());

// Run inference
float[][] input = {{...}};
float[][] output = new float[1][10];
tflite.run(input, output);
```

**Best Practices:**
- Use quantized models for better performance
- Leverage GPU delegate for acceleration
- Use NNAPI for hardware acceleration

#### 2. **iOS Deployment**
```swift
// Load TFLite model in iOS
guard let modelPath = Bundle.main.path(forResource: "model", ofType: "tflite") else {
    return
}
let interpreter = try Interpreter(modelPath: modelPath)

// Run inference
try interpreter.allocateTensors()
try interpreter.copy(inputData, toInputAt: 0)
try interpreter.invoke()
let outputTensor = try interpreter.output(at: 0)
```

**Best Practices:**
- Use Metal delegate for GPU acceleration
- Use Core ML delegate when possible
- Optimize for battery life

#### 3. **Embedded Systems (Raspberry Pi, etc.)**
```python
# Python on Raspberry Pi
import tflite_runtime.interpreter as tflite

interpreter = tflite.Interpreter(model_path="model.tflite")
interpreter.allocate_tensors()

# Run inference
interpreter.set_tensor(input_index, input_data)
interpreter.invoke()
output_data = interpreter.get_tensor(output_index)
```

#### 4. **Microcontrollers (TFLite Micro)**
- For very constrained devices
- C++ only
- Minimal memory footprint
- No dynamic memory allocation

In [ ]:
# Visualize deployment options
print("TensorFlow Deployment Options Summary:\n")
print("="*75)

deployment_options = [
    ("TF Serving (Docker)", "Server", "REST/gRPC", "High", "SavedModel"),
    ("Google Cloud AI", "Cloud", "REST API", "Auto", "SavedModel"),
    ("TFLite (Android)", "Mobile", "Native", "Medium", ".tflite"),
    ("TFLite (iOS)", "Mobile", "Native", "Medium", ".tflite"),
    ("TFLite (Raspberry Pi)", "Edge", "Python", "Low", ".tflite"),
    ("TFLite Micro", "MCU", "C++", "Very Low", ".tflite"),
    ("TensorFlow.js", "Browser", "JavaScript", "Medium", "tfjs"),
]

print(f"{'Platform':<25} {'Type':<10} {'API':<12} {'Scale':<10} {'Format':<15}")
print("-"*75)

for platform, ptype, api, scale, fmt in deployment_options:
    print(f"{platform:<25} {ptype:<10} {api:<12} {scale:<10} {fmt:<15}")

print("="*75)

print("\n💡 Choosing the Right Deployment:")
print("   • High throughput, server → TF Serving")
print("   • Managed, scalable → Google Cloud AI")
print("   • Mobile apps → TFLite (Android/iOS)")
print("   • Edge devices → TFLite on Linux")
print("   • Microcontrollers → TFLite Micro")
print("   • Web apps → TensorFlow.js")

## 15. Summary: Cloud and Mobile Deployment (25%-50%)

### What We've Covered

In this section, we explored deploying models to cloud and mobile platforms:

#### **Google Cloud AI Platform**
- **Managed Service**: No infrastructure management
- **Automatic Scaling**: Scales based on demand
- **Simple Deployment**: Upload model to GCS, create version
- **Monitoring**: Built-in logging and metrics
- **Cost**: Pay-per-prediction pricing
- **Global**: Deploy in multiple regions

#### **TensorFlow Lite**
- **Purpose**: Mobile and embedded deployment
- **Size Reduction**: 3-4x smaller than SavedModel
- **Efficient Inference**: Optimized for mobile hardware
- **Cross-Platform**: Android, iOS, embedded Linux
- **Offline**: Run models without network

#### **Quantization**
- **Weight Quantization**: 4x size reduction, minimal accuracy loss
- **Full Quantization**: Maximum optimization, requires calibration
- **Benefits**: Smaller size, faster inference, lower power
- **Trade-off**: Slight accuracy loss (usually < 1%)

### Key Takeaways

✅ **Cloud AI Platform** provides fully managed model serving  
✅ **TFLite** enables deployment on resource-constrained devices  
✅ **Quantization** dramatically reduces model size with minimal accuracy impact  
✅ **Multiple deployment options** for different use cases  
✅ **Optimization is crucial** for mobile and embedded deployment  

### Comparison Table

| Aspect | TF Serving | Cloud AI | TFLite | TFLite Quantized |
|--------|------------|----------|--------|------------------|
| **Size** | Large | Large | Medium | Small |
| **Speed** | Fast | Fast | Fast | Very Fast |
| **Device** | Server | Cloud | Mobile | Mobile/Edge |
| **Network** | Required | Required | Optional | Optional |
| **Cost** | Server | Pay-per-use | Free | Free |

### Next Steps

In the remaining sections, we'll cover:
- **GPU Acceleration**: Training on GPUs
- **Distributed Training**: Multi-GPU and multi-machine
- **Training Strategies**: Data and model parallelism
- **Google Cloud Training**: Large-scale training jobs

---

### Practice Exercises

1. **Convert your model** to TFLite format
2. **Apply quantization** and measure size reduction
3. **Test accuracy** of quantized model
4. **Explore Cloud AI Platform** (if you have GCP account)
5. **Compare inference speed** between formats

### Additional Resources

- [TensorFlow Lite Guide](https://www.tensorflow.org/lite/guide)
- [Post-Training Quantization](https://www.tensorflow.org/lite/performance/post_training_quantization)
- [Google Cloud AI Platform](https://cloud.google.com/ai-platform/docs)
- [TFLite Android Tutorial](https://www.tensorflow.org/lite/guide/android)
- [TFLite iOS Tutorial](https://www.tensorflow.org/lite/guide/ios)

# Part 3: GPU Acceleration and Distributed Training (Next 25%)

This section covers:
- **Using GPUs**: Hardware acceleration for training
- **GPU Management**: Memory management and device placement
- **Distributed Training**: Training across multiple GPUs/machines
- **Distribution Strategies**: TensorFlow's API for distributed training

---

## 16. Using GPUs for Training

### Why Use GPUs?

**Graphics Processing Units (GPUs)** dramatically accelerate deep learning training:

| Aspect | CPU | GPU |
|--------|-----|-----|
| **Cores** | 4-64 cores | 1,000s of cores |
| **Architecture** | General purpose | Parallel computation |
| **Speed (training)** | Baseline (1x) | 10-50x faster |
| **Best for** | Sequential tasks | Matrix operations |

### GPU Benefits for Deep Learning

1. **Massive Parallelism**: Thousands of cores for simultaneous computations
2. **Matrix Operations**: Optimized for neural network math
3. **Memory Bandwidth**: Fast data transfer
4. **Libraries**: Highly optimized (cuDNN, cuBLAS)

### Training Time Comparison

**Example**: Training ResNet-50 on ImageNet

- **CPU (Intel Xeon)**: ~2 weeks
- **Single GPU (NVIDIA V100)**: ~1 day
- **8 GPUs**: ~3 hours
- **256 GPUs**: ~15 minutes

### Getting Access to GPUs

#### Option 1: Purchase GPU Hardware
- **NVIDIA GPUs** for deep learning
- Requirements: CUDA Compute Capability 3.5+
- Popular models: RTX 3090, RTX 4090, A100

#### Option 2: Cloud GPU Services
- **Google Cloud Platform**: NVIDIA T4, V100, A100
- **Amazon AWS**: P3, P4 instances
- **Microsoft Azure**: NC, ND series
- **Lambda Labs**: GPU cloud service

#### Option 3: Google Colaboratory (Free!)
- Free GPU access (Tesla T4)
- Limited session time (12 hours max)
- Good for learning and small projects

Let's check GPU availability in this environment!

In [ ]:
# Check GPU availability
print("Checking GPU availability...\n")
print("="*70)

# Get list of physical devices
gpus = tf.config.list_physical_devices('GPU')
cpus = tf.config.list_physical_devices('CPU')

print(f"Number of CPUs available: {len(cpus)}")
print(f"Number of GPUs available: {len(gpus)}")

if gpus:
    print("\n✓ GPU(s) detected!")
    for i, gpu in enumerate(gpus):
        print(f"\nGPU {i}: {gpu}")
        print(f"  Device name: {gpu.name}")
        print(f"  Device type: {gpu.device_type}")
else:
    print("\n✗ No GPU detected")
    print("\nRunning on CPU. To use GPU:")
    print("  • Google Colab: Runtime → Change runtime type → GPU")
    print("  • Local: Install CUDA + cuDNN + tensorflow-gpu")
    print("  • Cloud: Use GPU-enabled VM instances")

print("\n" + "="*70)

In [ ]:
# Get detailed GPU information (if available)
print("Detailed GPU Information:\n")

if gpus:
    # Try to get GPU details using tensorflow
    try:
        gpu_details = tf.config.experimental.get_device_details(gpus[0])
        print("GPU Details:")
        for key, value in gpu_details.items():
            print(f"  {key}: {value}")
    except:
        print("  Detailed GPU info not available")
    
    # Check if TensorFlow was built with CUDA
    print(f"\nTensorFlow built with CUDA: {tf.test.is_built_with_cuda()}")
    
    # Try to get CUDA and cuDNN versions
    try:
        print(f"CUDA version: {tf.sysconfig.get_build_info()['cuda_version']}")
        print(f"cuDNN version: {tf.sysconfig.get_build_info()['cudnn_version']}")
    except:
        print("CUDA/cuDNN version info not available")
else:
    print("No GPU available - running on CPU")
    print("\nFor GPU access:")
    print("  1. Use Google Colab (free, easy)")
    print("  2. Use cloud services (AWS, GCP, Azure)")
    print("  3. Install local GPU with CUDA support")

## 17. Managing GPU Memory

### GPU Memory Challenges

GPUs have limited memory (typically 8-24 GB). Common issues:
- **Out of Memory (OOM)** errors
- **Memory fragmentation**
- **Multiple processes competing** for GPU

### Four Strategies for GPU Memory Management

#### 1. **Assign Specific GPUs to Processes**
Control which GPU each process uses via environment variables.

#### 2. **Set Memory Limits**
Allocate fixed amount of memory per process.

#### 3. **Dynamic Memory Growth**
Allow GPU memory to grow as needed (recommended).

#### 4. **Virtual Device Configuration**
Split one physical GPU into multiple virtual GPUs.

Let's implement each strategy!

In [ ]:
# Strategy 1: Assign specific GPUs to processes
# This is done via environment variables (before importing TensorFlow)

print("Strategy 1: Assign Specific GPUs to Processes\n")
print("="*70)
print("\nSet environment variables BEFORE importing TensorFlow:")
print()
print("# PowerShell (Windows):")
print("$env:CUDA_VISIBLE_DEVICES='0,1'")
print("python train_program_1.py")
print()
print("# Bash (Linux/Mac):")
print("CUDA_VISIBLE_DEVICES=0,1 python train_program_1.py")
print()
print("# Python code:")
print("import os")
print("os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'  # Use GPU 0 and 1")
print("import tensorflow as tf")
print()
print("Options:")
print("  '0'     - Use only GPU 0")
print("  '1'     - Use only GPU 1")
print("  '0,1'   - Use GPU 0 and 1")
print("  '2,3'   - Use GPU 2 and 3")
print("  '-1'    - Use CPU only")
print("  ''      - Use no GPUs")
print("\n" + "="*70)

In [ ]:
# Strategy 2: Set memory limit for GPU
print("Strategy 2: Set Memory Limit\n")
print("="*70)

if gpus:
    try:
        # Set memory limit to 2048 MB (2 GB) for first GPU
        tf.config.set_logical_device_configuration(
            gpus[0],
            [tf.config.LogicalDeviceConfiguration(memory_limit=2048)]
        )
        print("✓ Memory limit set to 2048 MB for GPU 0")
        print("\nNote: This must be done before any GPU operations")
    except RuntimeError as e:
        print(f"⚠ Cannot set memory limit after GPU initialization")
        print(f"Error: {e}")
    
    print("\nCode example:")
    print("gpus = tf.config.list_physical_devices('GPU')")
    print("if gpus:")
    print("    tf.config.set_logical_device_configuration(")
    print("        gpus[0],")
    print("        [tf.config.LogicalDeviceConfiguration(memory_limit=2048)]")
    print("    )")
else:
    print("No GPU available - this strategy requires GPU")
    print("\nExample code:")
    print("# Limit GPU to 2 GB")
    print("for gpu in tf.config.list_physical_devices('GPU'):")
    print("    tf.config.set_logical_device_configuration(")
    print("        gpu,")
    print("        [tf.config.LogicalDeviceConfiguration(memory_limit=2048)]")
    print("    )")

print("\n" + "="*70)

In [ ]:
# Strategy 3: Enable dynamic memory growth (RECOMMENDED)
print("Strategy 3: Dynamic Memory Growth (RECOMMENDED)\n")
print("="*70)

if gpus:
    try:
        # Enable memory growth for all GPUs
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        
        print(f"✓ Memory growth enabled for {len(gpus)} GPU(s)")
        print("\nWith memory growth:")
        print("  • TensorFlow allocates memory as needed")
        print("  • Starts with minimal allocation")
        print("  • Grows incrementally")
        print("  • Prevents one process from hogging all GPU memory")
        print("  • Allows multiple processes to share GPU")
        
    except RuntimeError as e:
        print(f"⚠ Cannot enable memory growth after initialization")
        print(f"Error: {e}")
        print("\nMemory growth must be set before TensorFlow uses GPU")
else:
    print("No GPU available")
    print("\nExample code for enabling memory growth:")
    print()
    print("gpus = tf.config.list_physical_devices('GPU')")
    print("if gpus:")
    print("    for gpu in gpus:")
    print("        tf.config.experimental.set_memory_growth(gpu, True)")

print("\n💡 Best Practice: Enable memory growth at start of program")
print("   Or set environment variable: TF_FORCE_GPU_ALLOW_GROWTH=true")

print("\n" + "="*70)

In [ ]:
# Strategy 4: Create virtual devices
print("Strategy 4: Split GPU into Virtual Devices\n")
print("="*70)

print("Split one physical GPU into multiple virtual GPUs:")
print("\nExample - Split GPU 0 into two 2GB virtual devices:")
print()
print("physical_gpus = tf.config.list_physical_devices('GPU')")
print("if physical_gpus:")
print("    tf.config.set_logical_device_configuration(")
print("        physical_gpus[0],")
print("        [")
print("            tf.config.LogicalDeviceConfiguration(memory_limit=2048),  # Virtual GPU 0")
print("            tf.config.LogicalDeviceConfiguration(memory_limit=2048)   # Virtual GPU 1")
print("        ]")
print("    )")
print()
print("Use cases:")
print("  • Testing multi-GPU code with single GPU")
print("  • Resource isolation between processes")
print("  • Development/debugging")
print()
print("⚠ Note: Virtual devices share the same physical hardware")
print("   Performance won't match true multi-GPU setups")

print("\n" + "="*70)

## 18. Device Placement and Parallel Execution

### Device Placement

TensorFlow automatically places operations on devices, but you can control placement manually:

**Default Behavior:**
- Variables and operations → GPU (if available)
- Operations without GPU support → CPU
- Automatic placement is usually optimal

**Manual Placement:**
Use `tf.device()` context to force specific device:

```python
with tf.device("/CPU:0"):
    # Operations run on CPU
    a = tf.constant([[1.0, 2.0], [3.0, 4.0]])
    b = tf.constant([[5.0, 6.0], [7.0, 8.0]])
    c = tf.matmul(a, b)
```

**Soft Placement:**
Automatically fall back to CPU if GPU unavailable:

```python
tf.config.set_soft_device_placement(True)
```

### Parallel Execution

TensorFlow automatically parallelizes operations:

#### On CPU:
- **Inter-op parallelism**: Run independent operations simultaneously
- **Intra-op parallelism**: Parallelize multi-threaded operations

#### On GPU:
- Operations execute sequentially
- Each operation uses highly parallel GPU kernels
- Multiple GPUs enable true parallelism

Let's demonstrate device placement!

In [ ]:
# Demonstrate device placement
import time

print("Device Placement Demo\n")
print("="*70)

# Enable soft placement (fallback to CPU if GPU unavailable)
tf.config.set_soft_device_placement(True)

# Test computation on different devices
def matrix_multiply_benchmark(device_name, size=1000):
    """Benchmark matrix multiplication on specified device"""
    with tf.device(device_name):
        # Create random matrices
        a = tf.random.normal([size, size])
        b = tf.random.normal([size, size])
        
        # Warm up
        _ = tf.matmul(a, b)
        
        # Benchmark
        start = time.time()
        for _ in range(10):
            c = tf.matmul(a, b)
        # Force execution
        _ = c.numpy()
        elapsed = time.time() - start
        
    return elapsed

# Test on available devices
print(f"Matrix multiplication benchmark ({10} iterations of {1000}x{1000}):\n")

# CPU test
cpu_time = matrix_multiply_benchmark("/CPU:0")
print(f"CPU:  {cpu_time:.4f} seconds")

# GPU test (if available)
if gpus:
    gpu_time = matrix_multiply_benchmark("/GPU:0")
    print(f"GPU:  {gpu_time:.4f} seconds")
    speedup = cpu_time / gpu_time
    print(f"\n💡 GPU is {speedup:.1f}x faster than CPU for this operation")
else:
    print("GPU: Not available")
    print("\n💡 GPU would be 10-100x faster for this operation")

print("\n" + "="*70)

## 19. Distributed Training Fundamentals

### Why Distributed Training?

As models and datasets grow, single-GPU training becomes insufficient:

| Challenge | Solution |
|-----------|----------|
| **Large models** | Split across devices (model parallelism) |
| **Large datasets** | Split data across devices (data parallelism) |
| **Long training time** | Use multiple GPUs/machines |

### Two Parallelism Strategies

#### 1. **Model Parallelism**
Split the model across multiple devices.

**Characteristics:**
- Each device has part of the model
- Complex to implement
- Architecture-specific
- High communication overhead
- Less common

**When to use:**
- Model too large for single GPU
- Specialized architectures (very deep/wide networks)

#### 2. **Data Parallelism** (RECOMMENDED)
Replicate model across devices, split data.

**Characteristics:**
- Each device has full model copy
- Easy to implement
- Works with any architecture
- Better scaling
- **Most common approach**

**How it works:**
1. Each device trains on different data batch
2. Gradients computed independently
3. Gradients aggregated (averaged)
4. Model weights updated synchronously

Let's visualize these strategies!

In [ ]:
# Visualize parallelism strategies
print("Parallelism Strategies Comparison\n")
print("="*70)

print("\n1. MODEL PARALLELISM")
print("-"*70)
print("┌──────────────┐  ┌──────────────┐  ┌──────────────┐")
print("│   GPU 0      │  │   GPU 1      │  │   GPU 2      │")
print("│              │  │              │  │              │")
print("│  Layer 1-3   │→ │  Layer 4-6   │→ │  Layer 7-9   │")
print("│              │  │              │  │              │")
print("└──────────────┘  └──────────────┘  └──────────────┘")
print("         ↓               ↓                ↓")
print("    Same Data       Same Data        Same Data")
print()
print("Characteristics:")
print("  • Model split across devices")
print("  • Sequential processing")
print("  • High communication overhead")
print("  • Complex implementation")

print("\n2. DATA PARALLELISM (RECOMMENDED)")
print("-"*70)
print("┌──────────────┐  ┌──────────────┐  ┌──────────────┐")
print("│   GPU 0      │  │   GPU 1      │  │   GPU 2      │")
print("│              │  │              │  │              │")
print("│ Full Model   │  │ Full Model   │  │ Full Model   │")
print("│  (copy)      │  │  (copy)      │  │  (copy)      │")
print("└──────────────┘  └──────────────┘  └──────────────┘")
print("         ↓               ↓                ↓")
print("    Batch 1         Batch 2          Batch 3")
print("         ↓               ↓                ↓")
print("    Gradients       Gradients        Gradients")
print("         └───────────────┴────────────────┘")
print("                         ↓")
print("                 Average & Update")
print()
print("Characteristics:")
print("  • Model replicated on each device")
print("  • Different data batches")
print("  • Synchronous gradient updates")
print("  • Easier to implement")

print("\n" + "="*70)

### Synchronous vs Asynchronous Training

In data parallelism, there are two update strategies:

#### **Synchronous Training**
- Wait for all workers to compute gradients
- Average all gradients
- Update model simultaneously
- **More stable convergence**
- Can use "backup workers" to avoid stragglers

#### **Asynchronous Training**
- Workers update model independently
- No waiting for slow workers
- **Faster iteration**
- Risk of stale gradients
- Less stable convergence

| Aspect | Synchronous | Asynchronous |
|--------|-------------|--------------|
| **Speed** | Slower (waits) | Faster (no wait) |
| **Stability** | More stable | Less stable |
| **Scaling** | Limited by slowest | Unlimited |
| **Use Case** | Most models | Sparse models |

**Recommendation**: Use synchronous training (MirroredStrategy) for most cases.

## 20. TensorFlow Distribution Strategies

### Distribution Strategies API

TensorFlow provides a **high-level API** for distributed training:

```python
strategy = tf.distribute.Strategy()
with strategy.scope():
    model = create_model()
    model.compile(...)
model.fit(...)
```

### Available Strategies

| Strategy | Use Case | Devices |
|----------|----------|---------|
| **MirroredStrategy** | Single machine, multi-GPU | Multiple GPUs |
| **CentralStorageStrategy** | Single machine, parameters on CPU | Multiple GPUs |
| **MultiWorkerMirroredStrategy** | Multi-machine | Multiple machines |
| **ParameterServerStrategy** | Large-scale async training | Many machines |
| **TPUStrategy** | Google TPU | TPU pods |

Let's implement distributed training with MirroredStrategy!

In [ ]:
# Implement MirroredStrategy for multi-GPU training
print("MirroredStrategy: Single-Machine Multi-GPU Training\n")
print("="*70)

# Check available GPUs
num_gpus = len(tf.config.list_physical_devices('GPU'))
print(f"Number of GPUs available: {num_gpus}")

if num_gpus > 0:
    # Create MirroredStrategy
    strategy = tf.distribute.MirroredStrategy()
    print(f"\n✓ MirroredStrategy created")
    print(f"Number of devices: {strategy.num_replicas_in_sync}")
    
    # Show device information
    print(f"\nDevices:")
    for i, device in enumerate(strategy.extended.worker_devices):
        print(f"  {i}: {device}")
else:
    # Fallback to default strategy (single device)
    strategy = tf.distribute.get_strategy()  # Default strategy
    print(f"\n⚠ No GPU available, using default strategy")
    print(f"Number of devices: {strategy.num_replicas_in_sync}")

print("\n" + "="*70)

In [ ]:
# Train model with distribution strategy
print("Training with Distribution Strategy\n")
print("="*70)

# Create model within strategy scope
with strategy.scope():
    # Build model
    distributed_model = keras.models.Sequential([
        keras.layers.Flatten(input_shape=[28, 28]),
        keras.layers.Dense(300, activation="relu"),
        keras.layers.Dense(100, activation="relu"),
        keras.layers.Dense(10, activation="softmax")
    ])
    
    # Compile model
    distributed_model.compile(
        optimizer="sgd",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    
    print("✓ Model created and compiled within strategy scope")

print(f"\nModel will be trained on {strategy.num_replicas_in_sync} device(s)")
print("\nKey points:")
print("  • Model is replicated across all devices")
print("  • Each device processes different data batch")
print("  • Gradients are averaged using AllReduce")
print("  • Weights updated synchronously")

print("\n" + "="*70)

In [ ]:
# Train the distributed model
print("Training distributed model...\n")

# Important: Batch size should be divisible by number of replicas
BATCH_SIZE_PER_REPLICA = 100
global_batch_size = BATCH_SIZE_PER_REPLICA * strategy.num_replicas_in_sync

print(f"Batch size per replica: {BATCH_SIZE_PER_REPLICA}")
print(f"Number of replicas: {strategy.num_replicas_in_sync}")
print(f"Global batch size: {global_batch_size}")
print()

# Train model (using subset for demo)
history_distributed = distributed_model.fit(
    X_train[:10000], y_train[:10000],
    epochs=3,
    batch_size=global_batch_size,
    validation_split=0.1,
    verbose=1
)

print("\n✓ Distributed training complete!")

# Evaluate
test_loss_dist, test_acc_dist = distributed_model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest accuracy: {test_acc_dist:.4f}")

### Other Distribution Strategies

Let's explore other available strategies:

#### **1. CentralStorageStrategy**
Parameters stored on CPU, computations on GPUs.

```python
strategy = tf.distribute.experimental.CentralStorageStrategy()
```

**Use case**: When GPU memory is limited

#### **2. MultiWorkerMirroredStrategy**
Distribute training across multiple machines.

```python
strategy = tf.distribute.MultiWorkerMirroredStrategy()
```

**Use case**: Very large models/datasets requiring multiple machines

Requires setting `TF_CONFIG` environment variable:
```python
import os, json
os.environ["TF_CONFIG"] = json.dumps({
    "cluster": {
        "worker": ["machine1:2222", "machine2:2222"]
    },
    "task": {"type": "worker", "index": 0}
})
```

#### **3. ParameterServerStrategy**
Asynchronous training with parameter servers.

```python
strategy = tf.distribute.experimental.ParameterServerStrategy()
```

**Use case**: Very large scale, sparse models

#### **4. TPUStrategy**
For Google's Tensor Processing Units.

```python
resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
tf.tpu.experimental.initialize_tpu_system(resolver)
strategy = tf.distribute.TPUStrategy(resolver)
```

**Use case**: Google Cloud TPU pods

In [ ]:
# Summary of distribution strategies
print("TensorFlow Distribution Strategies Summary\n")
print("="*75)

strategies = [
    ("MirroredStrategy", "Single machine", "2-8 GPUs", "Synchronous", "⭐ Most common"),
    ("CentralStorageStrategy", "Single machine", "2-8 GPUs", "Synchronous", "Limited GPU RAM"),
    ("MultiWorkerMirroredStrategy", "Multi-machine", "Many GPUs", "Synchronous", "Large scale"),
    ("ParameterServerStrategy", "Multi-machine", "Many workers", "Asynchronous", "Sparse models"),
    ("TPUStrategy", "Google Cloud", "TPU pods", "Synchronous", "Maximum speed"),
]

print(f"{'Strategy':<30} {'Scope':<15} {'Devices':<12} {'Sync':<14} {'Use Case':<20}")
print("-"*75)

for name, scope, devices, sync, use_case in strategies:
    print(f"{name:<30} {scope:<15} {devices:<12} {sync:<14} {use_case:<20}")

print("="*75)

print("\n💡 Quick Guide:")
print("   • Starting out? → MirroredStrategy")
print("   • Single machine, multiple GPUs? → MirroredStrategy")
print("   • Multiple machines needed? → MultiWorkerMirroredStrategy")
print("   • Huge scale (100s of workers)? → ParameterServerStrategy")
print("   • Google Cloud with TPUs? → TPUStrategy")

print("\n⚠ Important Notes:")
print("   • Always create model within strategy.scope()")
print("   • Batch size must be divisible by num_replicas")
print("   • Some layers may need modifications for distribution")
print("   • Save/load checkpoints carefully in distributed setting")

## 21. Summary: GPU Acceleration and Distributed Training (50%-75%)

### What We've Covered

In this section, we explored GPU acceleration and distributed training:

#### **GPU Training**
- **Massive Speedup**: 10-50x faster than CPU
- **Hardware Options**: Purchase GPU, cloud services, or Google Colab
- **CUDA Requirements**: NVIDIA GPU with CUDA Compute Capability 3.5+
- **Libraries Needed**: CUDA, cuDNN for optimal performance

#### **GPU Memory Management**
Four strategies for managing limited GPU memory:
1. **Assign GPUs**: `CUDA_VISIBLE_DEVICES` environment variable
2. **Set Limits**: Fixed memory allocation per GPU
3. **Dynamic Growth**: Memory grows as needed (recommended)
4. **Virtual Devices**: Split one GPU into multiple virtual GPUs

#### **Device Placement**
- **Automatic**: TensorFlow places operations optimally
- **Manual**: Use `tf.device()` for explicit control
- **Soft Placement**: Automatic fallback to CPU
- **Parallel Execution**: Automatic operation parallelization

#### **Distributed Training**
Two main approaches:
- **Model Parallelism**: Split model across devices (complex)
- **Data Parallelism**: Replicate model, split data (recommended)

Synchronization modes:
- **Synchronous**: Wait for all workers (stable, recommended)
- **Asynchronous**: No waiting (faster but less stable)

#### **Distribution Strategies API**
TensorFlow provides high-level API for distributed training:

| Strategy | Best For |
|----------|----------|
| **MirroredStrategy** | Single machine, 2-8 GPUs ⭐ |
| **CentralStorageStrategy** | Limited GPU memory |
| **MultiWorkerMirroredStrategy** | Multiple machines |
| **ParameterServerStrategy** | Large-scale async training |
| **TPUStrategy** | Google Cloud TPUs |

### Key Takeaways

✅ **GPUs dramatically accelerate training** (10-50x speedup)  
✅ **Memory management is crucial** for multi-process GPU usage  
✅ **Data parallelism** is easier and more common than model parallelism  
✅ **MirroredStrategy** is the go-to for multi-GPU training  
✅ **Distribution Strategies API** makes distributed training simple  
✅ **Batch size must scale** with number of devices  

### Best Practices

1. **Enable memory growth** at program start
2. **Use MirroredStrategy** for multi-GPU on single machine
3. **Scale batch size** proportionally to number of devices
4. **Create model within** `strategy.scope()`
5. **Use synchronous training** for most cases
6. **Monitor GPU utilization** with `nvidia-smi`

### Performance Expectations

**Typical speedups with data parallelism:**
- 2 GPUs: 1.8x speedup
- 4 GPUs: 3.5x speedup
- 8 GPUs: 6-7x speedup

**Scaling limitations:**
- Communication overhead
- Batch size constraints
- Synchronization costs

### What's Next?

In the final section (75%-100%), we'll cover:
- Training on Google Cloud AI Platform
- Hyperparameter tuning at scale
- TensorFlow.js for browser deployment
- Best practices and production tips

---

### Practice Exercises

1. **Check GPU availability** on your system
2. **Enable dynamic memory growth** in your code
3. **Benchmark** CPU vs GPU training speed
4. **Implement MirroredStrategy** for your model
5. **Experiment with batch sizes** for distributed training

### Additional Resources

- [TensorFlow GPU Guide](https://www.tensorflow.org/guide/gpu)
- [Distributed Training Guide](https://www.tensorflow.org/guide/distributed_training)
- [Distribution Strategies](https://www.tensorflow.org/guide/distributed_training#types_of_strategies)
- [NVIDIA CUDA](https://developer.nvidia.com/cuda-zone)
- [Google Colab](https://colab.research.google.com/)

# Part 4: Cloud Training and Advanced Deployment (Final 25%)

This final section covers:
- **Training on Google Cloud AI Platform**: Large-scale training jobs
- **Hyperparameter Tuning**: Automated optimization with Bayesian methods
- **TensorFlow.js**: Running models in web browsers
- **Production Best Practices**: Tips for real-world deployment

---

## 22. Training on Google Cloud AI Platform

### Why Use Cloud AI Platform for Training?

**Google Cloud AI Platform** (now part of Vertex AI) provides managed training infrastructure:

#### Benefits:
1. **No Infrastructure Management**: Focus on code, not servers
2. **Scalable Resources**: From single GPU to 100s of GPUs
3. **Pre-configured Environments**: TensorFlow, PyTorch pre-installed
4. **Distributed Training**: Built-in support
5. **Hyperparameter Tuning**: Automated optimization
6. **Cost Efficient**: Pay only for training time
7. **Integration**: Works with GCS, BigQuery, TensorBoard

#### Use Cases:
- Large models that don't fit on local GPU
- Very large datasets
- Hyperparameter search
- Distributed training experiments
- Regular retraining pipelines

### Training Job Architecture

```
Your Code (Python package)
         ↓
   Submit to AI Platform
         ↓
   Cloud Training Job
   (Multiple workers + parameter servers)
         ↓
   Trained Model → Google Cloud Storage
```

Let's explore how to submit training jobs!

### Preparing Training Code

To train on AI Platform, organize your code as a Python package:

#### Directory Structure:
```
my_training_package/
├── setup.py              # Package configuration
├── trainer/
│   ├── __init__.py
│   └── task.py          # Main training script
└── requirements.txt      # Dependencies
```

#### Key Requirements:
1. **Package Structure**: Must be installable Python package
2. **Entry Point**: Main training function in `task.py`
3. **Command-line Args**: For hyperparameters
4. **Cloud Storage**: Save model to GCS
5. **TensorBoard Logging**: For monitoring

#### Example `task.py`:
```python
import tensorflow as tf
import argparse

def train_model(args):
    # Load data from GCS
    dataset = tf.data.TFRecordDataset(f"gs://{args.bucket}/data.tfrecord")
    
    # Build model
    model = build_model(args.layers, args.units)
    
    # Train
    model.fit(dataset, epochs=args.epochs)
    
    # Save to GCS
    model.save(f"gs://{args.bucket}/trained_model")

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--bucket', type=str)
    parser.add_argument('--epochs', type=int, default=10)
    parser.add_argument('--layers', type=int, default=3)
    args = parser.parse_args()
    train_model(args)
```

In [ ]:
# Demonstrate submitting a training job to Google Cloud AI Platform
print("Submitting Training Job to Google Cloud AI Platform\n")
print("="*70)

# Configuration
JOB_NAME = "mnist_training_job_" + "20260108_120000"
REGION = "us-central1"
SCALE_TIER = "BASIC_GPU"  # Single GPU
RUNTIME_VERSION = "2.11"
PYTHON_VERSION = "3.9"
PACKAGE_PATH = "./trainer"
MODULE_NAME = "trainer.task"
STAGING_BUCKET = "gs://my-ml-bucket/staging"
JOB_DIR = "gs://my-ml-bucket/jobs/" + JOB_NAME

print("Job Configuration:")
print(f"  Job Name:        {JOB_NAME}")
print(f"  Region:          {REGION}")
print(f"  Scale Tier:      {SCALE_TIER}")
print(f"  Runtime Version: {RUNTIME_VERSION}")
print(f"  Python Version:  {PYTHON_VERSION}")
print(f"  Job Directory:   {JOB_DIR}")

print("\n" + "="*70)
print("\nCommand to submit job:\n")

cmd = f"""gcloud ai-platform jobs submit training {JOB_NAME} \\
    --region={REGION} \\
    --scale-tier={SCALE_TIER} \\
    --runtime-version={RUNTIME_VERSION} \\
    --python-version={PYTHON_VERSION} \\
    --package-path={PACKAGE_PATH} \\
    --module-name={MODULE_NAME} \\
    --staging-bucket={STAGING_BUCKET} \\
    --job-dir={JOB_DIR} \\
    -- \\
    --epochs=10 \\
    --learning-rate=0.001 \\
    --batch-size=128"""

print(cmd)

print("\n" + "="*70)

In [ ]:
# Scale Tiers: Different machine configurations
print("Google Cloud AI Platform Scale Tiers\n")
print("="*70)

scale_tiers = [
    ("BASIC", "Single worker", "No GPUs", "Development/testing"),
    ("BASIC_GPU", "Single worker", "1 GPU (NVIDIA Tesla K80)", "Small models"),
    ("BASIC_TPU", "Single worker", "1 Cloud TPU", "TPU experiments"),
    ("STANDARD_1", "Multiple workers", "No GPUs", "CPU distributed"),
    ("PREMIUM_1", "20 workers", "11 parameter servers", "Large distributed"),
    ("CUSTOM", "User-defined", "Custom config", "Specific needs"),
]

print(f"{'Tier':<15} {'Workers':<20} {'Accelerators':<25} {'Use Case':<20}")
print("-"*70)

for tier, workers, accel, use_case in scale_tiers:
    print(f"{tier:<15} {workers:<20} {accel:<25} {use_case:<20}")

print("="*70)

print("\n💡 Choosing Scale Tier:")
print("   • Development → BASIC")
print("   • Single GPU needed → BASIC_GPU")
print("   • Multi-GPU training → PREMIUM_1")
print("   • Custom requirements → CUSTOM")
print()
print("⚠ Cost increases with scale tier:")
print("   • BASIC: ~$0.05/hour")
print("   • BASIC_GPU: ~$0.70/hour")
print("   • PREMIUM_1: ~$10/hour")
print()
print("🔗 Pricing: https://cloud.google.com/ai-platform/training/pricing")

## 23. Hyperparameter Tuning at Scale

### Why Automated Hyperparameter Tuning?

Finding optimal hyperparameters manually is:
- **Time-consuming**: Many experiments needed
- **Expensive**: Trial and error wastes resources
- **Suboptimal**: May miss best configuration

**Solution**: Automated hyperparameter tuning with **Bayesian Optimization**

### How It Works

Google Cloud AI Platform uses **Google Vizier**:

1. **Bayesian Optimization**: Builds probabilistic model of hyperparameter space
2. **Smart Sampling**: Focuses on promising regions
3. **Parallel Trials**: Runs multiple experiments simultaneously
4. **Early Stopping**: Stops unpromising trials early

**Benefits vs Grid/Random Search:**
- Fewer trials needed
- Finds better configurations
- Cost-effective
- Automatic

### Hyperparameter Configuration

Create a `config.yaml` file:

```yaml
trainingInput:
  scaleTier: BASIC_GPU
  hyperparameters:
    goal: MAXIMIZE
    hyperparameterMetricTag: accuracy
    maxTrials: 20
    maxParallelTrials: 4
    params:
      - parameterName: learning_rate
        type: DOUBLE
        minValue: 0.0001
        maxValue: 0.1
        scaleType: UNIT_LOG_SCALE
      - parameterName: num_layers
        type: INTEGER
        minValue: 2
        maxValue: 5
        scaleType: UNIT_LINEAR_SCALE
      - parameterName: units_per_layer
        type: DISCRETE
        discreteValues: [64, 128, 256, 512]
```

Let's create an example configuration!

In [ ]:
# Create hyperparameter tuning configuration
import yaml

print("Hyperparameter Tuning Configuration\n")
print("="*70)

# Define hyperparameter tuning config
tuning_config = {
    'trainingInput': {
        'scaleTier': 'BASIC_GPU',
        'hyperparameters': {
            'goal': 'MAXIMIZE',
            'hyperparameterMetricTag': 'accuracy',
            'maxTrials': 20,
            'maxParallelTrials': 4,
            'enableTrialEarlyStopping': True,
            'params': [
                {
                    'parameterName': 'learning_rate',
                    'type': 'DOUBLE',
                    'minValue': 0.0001,
                    'maxValue': 0.1,
                    'scaleType': 'UNIT_LOG_SCALE'
                },
                {
                    'parameterName': 'num_layers',
                    'type': 'INTEGER',
                    'minValue': 2,
                    'maxValue': 5,
                    'scaleType': 'UNIT_LINEAR_SCALE'
                },
                {
                    'parameterName': 'dropout_rate',
                    'type': 'DOUBLE',
                    'minValue': 0.1,
                    'maxValue': 0.5,
                    'scaleType': 'UNIT_LINEAR_SCALE'
                },
                {
                    'parameterName': 'units_per_layer',
                    'type': 'DISCRETE',
                    'discreteValues': [64, 128, 256, 512]
                }
            ]
        }
    }
}

# Display configuration
print("Configuration:")
print(yaml.dump(tuning_config, default_flow_style=False))

print("\n" + "="*70)
print("\nTo submit tuning job:")
print()
print("gcloud ai-platform jobs submit training my_tuning_job \\")
print("    --config=config.yaml \\")
print("    --package-path=./trainer \\")
print("    --module-name=trainer.task")

print("\n" + "="*70)

In [ ]:
# Explain scale types for hyperparameter tuning
print("Hyperparameter Scale Types\n")
print("="*70)

print("\n1. UNIT_LINEAR_SCALE")
print("-"*70)
print("Uniform distribution across range")
print("Use when: No prior knowledge about optimal value")
print("Example: Number of layers (2-5)")
print()
print("Sampling: [2, 2.5, 3, 3.5, 4, 4.5, 5]")

print("\n2. UNIT_LOG_SCALE")
print("-"*70)
print("Logarithmic distribution - more samples near minimum")
print("Use when: Optimal value likely closer to maximum")
print("Example: Learning rate (0.0001 - 0.1)")
print()
print("Sampling: [0.0001, 0.001, 0.01, 0.1]")
print("         (powers of 10)")

print("\n3. UNIT_REVERSE_LOG_SCALE")
print("-"*70)
print("Reverse logarithmic - more samples near maximum")
print("Use when: Optimal value likely closer to minimum")
print("Example: Dropout rate (higher is often better)")

print("\n4. DISCRETE")
print("-"*70)
print("Specific values only")
print("Use when: Limited set of valid values")
print("Example: units_per_layer = [64, 128, 256, 512]")

print("\n" + "="*70)

print("\n💡 Best Practices:")
print("   • Use UNIT_LOG_SCALE for learning rate")
print("   • Use UNIT_LINEAR_SCALE for architecture params")
print("   • Use DISCRETE for categorical choices")
print("   • Enable early stopping to save costs")
print("   • Start with 10-20 trials for initial exploration")
print("   • Monitor TensorBoard for trial progress")

## 24. TensorFlow.js: Models in the Browser

### What is TensorFlow.js?

**TensorFlow.js** enables running ML models directly in web browsers using JavaScript.

#### Use Cases:
1. **Privacy**: Data never leaves the user's device
2. **Low Latency**: No server round-trip
3. **Offline**: Works without internet
4. **Interactive**: Real-time predictions in browser
5. **Cost**: No server inference costs

#### Popular Applications:
- **Pose Detection**: Body tracking in browser
- **Image Classification**: Drag-and-drop image recognition
- **Text Generation**: Client-side text completion
- **Style Transfer**: Real-time artistic effects
- **Face Detection**: Camera-based applications

### Architecture

```
Python Model (TensorFlow/Keras)
         ↓
  TensorFlow.js Converter
         ↓
   TF.js Model Files
   (model.json + weights)
         ↓
   Browser (JavaScript)
         ↓
   WebGL Acceleration
```

### Performance

**Execution Backends:**
- **WebGL**: GPU acceleration (fastest)
- **WebAssembly**: CPU with SIMD
- **CPU**: JavaScript fallback

**Typical Speed:**
- Simple models: Real-time (30+ FPS)
- Complex models: 5-10 FPS
- Mobile browsers: 3-5 FPS

Let's convert our model to TF.js format!

In [ ]:
# Install TensorFlow.js converter
print("Installing TensorFlow.js converter...\n")
!{sys.executable} -m pip install -q tensorflowjs

print("Installation complete!")
print("\n" + "="*70)

# Convert model to TensorFlow.js format
print("\nConverting model to TensorFlow.js format...")

import tensorflowjs as tfjs

# Output directory for TF.js model
tfjs_output_dir = "tfjs_model"

# Convert the model
tfjs.converters.save_keras_model(model, tfjs_output_dir)

print(f"✓ Model converted and saved to: {tfjs_output_dir}")

# Display generated files
import os
print(f"\nGenerated files:")
for file in os.listdir(tfjs_output_dir):
    file_path = os.path.join(tfjs_output_dir, file)
    size = os.path.getsize(file_path) / 1024  # KB
    print(f"  • {file:<30} ({size:>8.2f} KB)")

print("\n" + "="*70)

In [ ]:
# Show JavaScript code for using the model
print("Using TensorFlow.js Model in Browser\n")
print("="*70)

html_code = """
<!DOCTYPE html>
<html>
<head>
    <title>MNIST Digit Recognition</title>
    <!-- Load TensorFlow.js -->
    <script src="https://cdn.jsdelivr.net/npm/@tensorflow/tfjs"></script>
</head>
<body>
    <h1>MNIST Digit Recognition</h1>
    
    <!-- Canvas for drawing -->
    <canvas id="canvas" width="280" height="280" 
            style="border: 2px solid black;"></canvas>
    <br>
    <button onclick="predict()">Predict</button>
    <button onclick="clearCanvas()">Clear</button>
    
    <h2>Prediction: <span id="prediction">-</span></h2>
    <div id="confidence"></div>
    
    <script>
        let model;
        
        // Load the model
        async function loadModel() {
            model = await tf.loadLayersModel('tfjs_model/model.json');
            console.log('Model loaded!');
        }
        
        // Predict digit
        async function predict() {
            // Get canvas data
            const canvas = document.getElementById('canvas');
            const ctx = canvas.getContext('2d');
            
            // Preprocess: resize to 28x28, normalize
            let img = tf.browser.fromPixels(canvas, 1);
            img = tf.image.resizeBilinear(img, [28, 28]);
            img = img.div(255.0);
            img = img.expandDims(0);  // Add batch dimension
            
            // Make prediction
            const prediction = await model.predict(img);
            const probabilities = await prediction.data();
            
            // Get predicted class
            const predictedClass = prediction.argMax(-1).dataSync()[0];
            
            // Display results
            document.getElementById('prediction').textContent = predictedClass;
            
            // Show all probabilities
            let html = '<h3>Confidence:</h3>';
            for (let i = 0; i < 10; i++) {
                const prob = (probabilities[i] * 100).toFixed(2);
                html += `Digit ${i}: ${prob}%<br>`;
            }
            document.getElementById('confidence').innerHTML = html;
        }
        
        // Clear canvas
        function clearCanvas() {
            const canvas = document.getElementById('canvas');
            const ctx = canvas.getContext('2d');
            ctx.fillStyle = 'white';
            ctx.fillRect(0, 0, 280, 280);
        }
        
        // Initialize
        loadModel();
        clearCanvas();
        
        // Setup drawing
        const canvas = document.getElementById('canvas');
        const ctx = canvas.getContext('2d');
        let drawing = false;
        
        canvas.onmousedown = () => drawing = true;
        canvas.onmouseup = () => drawing = false;
        canvas.onmousemove = (e) => {
            if (!drawing) return;
            ctx.fillStyle = 'black';
            ctx.fillRect(e.offsetX-5, e.offsetY-5, 10, 10);
        };
    </script>
</body>
</html>
"""

print("Complete HTML Example:")
print("-"*70)
print(html_code[:500] + "\n... (truncated)")
print()
print("Save this to 'index.html' and serve with:")
print("  python -m http.server 8000")
print()
print("Then open: http://localhost:8000")

print("\n" + "="*70)

In [ ]:
# Compare model formats
print("Model Format Comparison\n")
print("="*75)

formats = [
    ("SavedModel", "Python/TF Serving", "Server", "Large", "Fast", "High"),
    ("TFLite", "Mobile/Embedded", "Device", "Small", "Fast", "Medium"),
    ("TFLite Quantized", "Mobile/IoT", "Device", "Very Small", "Very Fast", "Medium"),
    ("TensorFlow.js", "Browser", "Browser", "Medium", "Medium", "Low-Med"),
]

print(f"{'Format':<20} {'Platform':<20} {'Deploy':<10} {'Size':<12} {'Speed':<10} {'Accuracy'}")
print("-"*75)

for fmt, platform, deploy, size, speed, acc in formats:
    print(f"{fmt:<20} {platform:<20} {deploy:<10} {size:<12} {speed:<10} {acc}")

print("="*75)

print("\n💡 Choosing the Right Format:")
print()
print("┌─────────────────────────────────────────────────────────────┐")
print("│  Use Case                    →  Format                      │")
print("├─────────────────────────────────────────────────────────────┤")
print("│  Production API server       →  SavedModel + TF Serving    │")
print("│  Cloud deployment            →  SavedModel + Cloud AI      │")
print("│  Mobile app (Android/iOS)    →  TFLite (quantized)         │")
print("│  Edge device (Raspberry Pi)  →  TFLite                     │")
print("│  Microcontroller             →  TFLite Micro               │")
print("│  Web application             →  TensorFlow.js              │")
print("│  Browser-based demo          →  TensorFlow.js              │")
print("└─────────────────────────────────────────────────────────────┘")

print("\n⚠ Trade-offs:")
print("   • Size ↔ Accuracy: Smaller models may lose accuracy")
print("   • Speed ↔ Size: Quantization improves both")
print("   • Flexibility ↔ Performance: Specialized formats are faster")
print("   • Browser ↔ Server: Browser has privacy benefits but slower")

## 25. Production Best Practices

### Deployment Checklist

Before deploying models to production, ensure:

#### **1. Model Quality**
- ✅ Validate on holdout test set
- ✅ Check for overfitting
- ✅ Test edge cases
- ✅ Measure inference latency
- ✅ Document expected accuracy

#### **2. Performance Optimization**
- ✅ Optimize batch size
- ✅ Use mixed precision (float16)
- ✅ Enable XLA compilation
- ✅ Quantize for mobile/edge
- ✅ Profile bottlenecks

#### **3. Monitoring**
- ✅ Log predictions
- ✅ Monitor latency
- ✅ Track error rates
- ✅ Detect data drift
- ✅ Alert on anomalies

#### **4. Security**
- ✅ Validate inputs
- ✅ Rate limiting
- ✅ Authentication/authorization
- ✅ Encrypt data in transit
- ✅ Audit logging

#### **5. Versioning**
- ✅ Version models systematically
- ✅ Track model lineage
- ✅ Enable rollback
- ✅ A/B test new versions
- ✅ Document changes

### Common Pitfalls to Avoid

1. **Training-Serving Skew**: Ensure preprocessing is identical
2. **Data Drift**: Monitor input distribution changes
3. **Memory Leaks**: Properly manage resources
4. **Cold Start**: Pre-warm models
5. **Batch Size Mismatch**: Test with production batch sizes
6. **Missing Error Handling**: Handle edge cases gracefully

Let's create a deployment checklist!

In [ ]:
# Production deployment checklist
print("TensorFlow Production Deployment Checklist\n")
print("="*70)

checklist = {
    "Model Quality": [
        "Validate on holdout test set",
        "Check for overfitting (train vs val loss)",
        "Test with edge cases and adversarial examples",
        "Measure and document inference latency",
        "Benchmark accuracy against baseline",
        "Test with various batch sizes"
    ],
    "Performance": [
        "Profile model for bottlenecks",
        "Optimize batch size for throughput",
        "Enable GPU/TPU if available",
        "Use mixed precision training (float16)",
        "Enable XLA compilation for speed",
        "Quantize models for mobile/edge"
    ],
    "Infrastructure": [
        "Set up monitoring (Prometheus, Grafana)",
        "Configure logging (ELK, Stackdriver)",
        "Implement health checks",
        "Set up auto-scaling",
        "Configure load balancing",
        "Plan for disaster recovery"
    ],
    "Security": [
        "Validate and sanitize inputs",
        "Implement rate limiting",
        "Add authentication/authorization",
        "Encrypt data in transit (TLS)",
        "Encrypt sensitive data at rest",
        "Audit log all predictions"
    ],
    "MLOps": [
        "Version models systematically (0001, 0002...)",
        "Track model lineage and metadata",
        "Implement canary deployments",
        "Enable easy rollback",
        "A/B test new models",
        "Document model cards"
    ],
    "Monitoring": [
        "Monitor prediction latency (p50, p95, p99)",
        "Track error rates and types",
        "Detect data drift",
        "Monitor resource usage (CPU, GPU, RAM)",
        "Set up alerts for anomalies",
        "Create dashboards"
    ]
}

for category, items in checklist.items():
    print(f"\n{category.upper()}")
    print("-"*70)
    for i, item in enumerate(items, 1):
        print(f"  {i}. ☐ {item}")

print("\n" + "="*70)
print("\n💡 Priority Order:")
print("   1️⃣  Model Quality → Must be accurate")
print("   2️⃣  Security → Protect users and system")
print("   3️⃣  Monitoring → Know when things break")
print("   4️⃣  Performance → Optimize after it works")
print("   5️⃣  MLOps → Enable iteration")

print("\n" + "="*70)

In [ ]:
# Quick reference: TensorFlow deployment commands
print("TensorFlow Deployment Quick Reference\n")
print("="*70)

commands = {
    "SavedModel": [
        ("Export model", "tf.saved_model.save(model, 'path/to/model')"),
        ("Load model", "model = tf.keras.models.load_model('path/to/model')"),
        ("Inspect model", "saved_model_cli show --dir path/to/model --all"),
    ],
    "TF Serving (Docker)": [
        ("Pull image", "docker pull tensorflow/serving"),
        ("Run server", "docker run -p 8501:8501 -v model:/models/model -e MODEL_NAME=model tensorflow/serving"),
        ("Query REST", "curl -X POST http://localhost:8501/v1/models/model:predict -d '{...}'"),
    ],
    "TensorFlow Lite": [
        ("Convert model", "converter = tf.lite.TFLiteConverter.from_saved_model(path)"),
        ("Quantize", "converter.optimizations = [tf.lite.Optimize.DEFAULT]"),
        ("Save model", "with open('model.tflite', 'wb') as f: f.write(tflite_model)"),
    ],
    "TensorFlow.js": [
        ("Install", "pip install tensorflowjs"),
        ("Convert", "tensorflowjs_converter --input_format=keras model.h5 tfjs/"),
        ("Load in JS", "const model = await tf.loadLayersModel('tfjs/model.json')"),
    ],
    "Google Cloud": [
        ("Upload to GCS", "gsutil cp -r model/ gs://bucket/model/"),
        ("Create model", "gcloud ai-platform models create model_name"),
        ("Deploy version", "gcloud ai-platform versions create v1 --model model_name --origin gs://..."),
    ],
    "Distribution": [
        ("Multi-GPU", "strategy = tf.distribute.MirroredStrategy()"),
        ("GPU memory", "tf.config.experimental.set_memory_growth(gpu, True)"),
        ("Device placement", "with tf.device('/GPU:0'): ..."),
    ],
}

for category, cmds in commands.items():
    print(f"\n{category}")
    print("-"*70)
    for desc, cmd in cmds:
        print(f"  • {desc}:")
        print(f"    {cmd}")

print("\n" + "="*70)

## 26. Final Summary: Complete Chapter 19 (100%)

### 🎉 Congratulations!

You've completed the comprehensive tutorial on **Training and Deploying TensorFlow Models at Scale**!

---

### 📚 What We Covered

#### **Part 1: Model Serving (0-25%)**
- ✅ TensorFlow Serving with Docker
- ✅ SavedModel format and inspection
- ✅ REST and gRPC APIs
- ✅ Model versioning and deployment

#### **Part 2: Cloud and Mobile (25-50%)**
- ✅ Google Cloud AI Platform deployment
- ✅ TensorFlow Lite conversion
- ✅ Quantization techniques
- ✅ Mobile and embedded deployment

#### **Part 3: GPU and Distribution (50-75%)**
- ✅ GPU acceleration basics
- ✅ GPU memory management
- ✅ Distributed training strategies
- ✅ Data vs model parallelism

#### **Part 4: Advanced Topics (75-100%)**
- ✅ Cloud training jobs
- ✅ Automated hyperparameter tuning
- ✅ TensorFlow.js for browsers
- ✅ Production best practices

---

### 🎯 Key Takeaways

1. **TensorFlow Serving** provides production-ready model deployment with versioning
2. **gRPC is faster** than REST for large data transfers
3. **TFLite and quantization** enable mobile/embedded deployment
4. **GPUs accelerate training** by 10-50x
5. **MirroredStrategy** is the go-to for multi-GPU training
6. **Google Cloud AI Platform** handles infrastructure at scale
7. **Bayesian optimization** improves hyperparameter search
8. **TensorFlow.js** enables browser-based ML
9. **Monitoring and versioning** are critical for production

---

### 🚀 Deployment Decision Tree

```
Need to deploy a model?
│
├─ Server/API?
│  ├─ Small scale → TF Serving (Docker)
│  └─ Large scale → Google Cloud AI Platform
│
├─ Mobile app?
│  └─ Use TFLite (with quantization)
│
├─ Web browser?
│  └─ Use TensorFlow.js
│
└─ Edge device?
   ├─ Raspberry Pi → TFLite
   └─ Microcontroller → TFLite Micro
```

---

### 📊 Performance Comparison

| Deployment | Latency | Throughput | Cost | Ease |
|------------|---------|------------|------|------|
| **TF Serving** | Low | High | Medium | Medium |
| **Cloud AI** | Medium | Very High | High | Easy |
| **TFLite** | Very Low | Medium | Free | Medium |
| **TensorFlow.js** | Medium | Low | Free | Easy |

---

### 🛠️ Tools You've Learned

- `tf.saved_model.save()` - Export models
- `saved_model_cli` - Inspect models
- `tf.lite.TFLiteConverter` - Mobile conversion
- `tf.distribute.MirroredStrategy()` - Multi-GPU training
- `gcloud ai-platform` - Cloud training/serving
- `tensorflowjs_converter` - Browser deployment
- Docker for TF Serving
- Google Cloud Storage integration

---

### 💡 Best Practices Summary

#### **Development**
- Start simple, optimize later
- Use Google Colab for free GPUs
- Version everything (code, data, models)
- Test locally before cloud deployment

#### **Training**
- Enable GPU memory growth
- Use MirroredStrategy for multi-GPU
- Scale batch size with number of devices
- Monitor with TensorBoard

#### **Deployment**
- Validate models thoroughly
- Implement monitoring from day 1
- Plan for model updates
- Document everything

#### **Production**
- Use canary deployments
- Implement rollback mechanisms
- Monitor data drift
- Log predictions for debugging

---

### 📖 Further Learning

#### **Official Resources**
- [TensorFlow Serving Guide](https://www.tensorflow.org/tfx/guide/serving)
- [TFLite Guide](https://www.tensorflow.org/lite)
- [Distributed Training](https://www.tensorflow.org/guide/distributed_training)
- [TensorFlow.js](https://www.tensorflow.org/js)

#### **Cloud Platforms**
- [Google Cloud AI Platform](https://cloud.google.com/ai-platform)
- [AWS SageMaker](https://aws.amazon.com/sagemaker/)
- [Azure ML](https://azure.microsoft.com/en-us/services/machine-learning/)

#### **Books**
- "Machine Learning Engineering" by Andriy Burkov
- "Building Machine Learning Powered Applications" by Emmanuel Ameisen
- "Designing Data-Intensive Applications" by Martin Kleppmann

---

### 🎓 Practice Projects

1. **Deploy MNIST to TF Serving**: Complete REST and gRPC integration
2. **Build Mobile App**: Create Android/iOS app with TFLite
3. **Browser Demo**: Deploy model to web with TensorFlow.js
4. **Multi-GPU Training**: Train large model with MirroredStrategy
5. **Cloud Pipeline**: Complete training → serving pipeline on GCP
6. **Hyperparameter Tuning**: Optimize model with AI Platform tuning
7. **Production System**: Build monitored, versioned deployment

---

### 🌟 Final Thoughts

> *"My greatest hope is that this book will inspire you to build a wonderful ML application that will benefit all of us!"*  
> — Aurélien Géron

You now have the knowledge to:
- Train models efficiently on GPUs
- Deploy models to production
- Optimize for mobile and edge devices
- Scale training across multiple machines
- Monitor and maintain ML systems

**The journey doesn't end here!** Keep building, deploying, and iterating on your ML models.

---

### ✅ Chapter 19 Complete!

Thank you for working through this comprehensive tutorial. You're now equipped to deploy TensorFlow models at scale across any platform!

**Next steps:**
- Practice with your own models
- Explore cloud platforms
- Build production systems
- Share your knowledge

**Happy deploying! 🚀🎉**